<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_2/Classical_Text_Vectorization_One_Hot_Encoding_to_Latent_Semantic_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Классические методы векторизации текста: от one-hot encoding до латентного семантического анализа

## Введение

Любая задача обработки естественного языка (NLP) — классификация документов, кластеризация, информационный поиск, машинный перевод — требует перевода текста в числовую форму, пригодную для математических моделей и алгоритмов машинного обучения. Выбор способа такого представления напрямую определяет, какие закономерности языка сможет уловить модель, насколько компактным и вычислительно эффективным будет векторное пространство и как оно поведёт себя при работе с шумом и редкими словами.

В этой серии лекций рассматривается эволюция **классических** методов векторизации текста — от простейшего one-hot encoding до вероятностной тематической модели латентного размещения Дирихле (LDA). Каждый метод излагается по единой схеме: математическая формализация, наглядный числовой пример на фиксированном учебном корпусе из трёх документов и критический анализ ограничений. Такой подход позволяет наглядно проследить, как усложнение моделей приводит к более содержательным и компактным представлениям текста.

Изучение классических подходов необходимо для понимания принципов работы современных нейросетевых эмбеддингов (Word2Vec, BERT и др.), которые будут рассмотрены в следующих сериях.

---

## Часть 1. Введение в методы векторизации текста и one-hot encoding

### 1.1 Зачем нужна векторизация текста

Текст в естественном виде — последовательность символов или слов — не может быть непосредственно передан в алгоритмы машинного обучения, которые оперируют числами. Поэтому первым шагом любого NLP-проекта является **векторизация** — отображение текстовых единиц (слов, предложений, документов) в числовые векторы, максимально сохраняющие содержательную информацию исходного текста.

Понятие векторизации охватывает широкий спектр методов, различающихся по сложности, вычислительной эффективности и способности улавливать семантические отношения. Историческое развитие этих методов можно представить как последовательное преодоление ограничений предыдущих подходов:

1. **One-hot encoding** — самый простой способ, который лишь уникально идентифицирует слово, полностью игнорируя его смысл.
2. **Частотные модели документов (Bag of Words, TF-IDF)** — учитывают статистическую значимость терминов, но не улавливают смысловую близость слов.
3. **Латентные методы (LSA, тематические модели)** — выявляют скрытые факторы и позволяют словам и документам находиться в общем непрерывном пространстве.
4. **Нейросетевые эмбеддинги (Word2Vec, BERT)** — дают плотные семантически насыщенные представления, обучаемые на огромных корпусах.

В этой части мы детально разберём первый, базовый способ представления слова — one-hot encoding. Несмотря на кажущуюся примитивность, он важен для понимания того, почему потребовались более сложные подходы и какие принципиальные проблемы возникают при работе с естественным языком.

### 1.2 Понятие словаря и индексного пространства

Пусть имеется некоторая коллекция текстов — **корпус**. Корпус может состоять из одного или нескольких документов, а документы — из последовательности слов. Под словом понимается лексическая единица после предварительной обработки: приведения к нижнему регистру, удаления знаков препинания, лемматизации или стемминга. Для простоты будем считать, что слова уже нормализованы и представляют собой минимальные смысловые единицы, разделённые пробелами.

Из всего корпуса извлекается множество уникальных слов — **словарь** $V$. Его размер $|V|$ (или $N$) может варьироваться от нескольких десятков в учебных примерах до нескольких миллионов в реальных корпусах. Каждому слову $w \in V$ присваивается уникальный целочисленный индекс $i \in \{1, 2, \ldots, N\}$. Такое соответствие задаётся биективной функцией индексирования:
$$\text{id}: V \to \{1, 2, \ldots, N\}.$$
Порядок присвоения индексов может быть произвольным, но на практике часто используют сортировку по алфавиту или по убыванию частоты встречаемости.

Когда все слова корпуса заменяются их индексами, текст превращается в последовательность целых чисел. Однако такое представление неудобно для большинства алгоритмов машинного обучения, поскольку числа вводят искусственный порядок, не отражающий семантических отношений. Например, если слову «кошка» присвоен индекс 1, а слову «собака» индекс 2, то разность индексов может быть ошибочно истолкована как мера близости. Поэтому возникает необходимость в векторном представлении, в котором каждое слово было бы точкой в некотором векторном пространстве. Простейшим из таких представлений является one-hot encoding.



## 1.3 Определение one-hot encoding

One-hot encoding, или унитарное кодирование, сопоставляет каждому слову $w_i$ из словаря $V$ вектор $e_i$ размерности $N = |V|$. Все компоненты этого вектора равны нулю, за исключением одной — той, которая соответствует индексу данного слова. Если зафиксировать порядок слов в словаре, например, $V = \{w_1, w_2, \ldots, w_N\}$, то вектор для слова $w_i$ имеет вид:

$$
e_i = \left( 0, \; 0, \; \ldots, \; 0, \; \underbrace{1}_{i\text{-я позиция}}, \; 0, \; \ldots, \; 0 \right)^\top ,
$$

где символ $\top$ обозначает транспонирование, то есть вектор записан как столбец, но для удобства часто изображается строкой.

Формально:

$$
(e_i)_j =
\begin{cases}
1, & \text{если } j = i, \\
0, & \text{если } j \neq i.
\end{cases}
$$

Такой вектор содержит ровно одну единицу и $N-1$ нулей. Матрица, составленная из всех one-hot векторов словаря, является единичной матрицей размером $N \times N$:

$$
E = \begin{pmatrix}
1 & 0 & \cdots & 0 \\
0 & 1 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & 1
\end{pmatrix} = I_N .
$$

Это свойство проистекает из ортонормированности системы векторов: для любых $i, j$ скалярное произведение

$$
e_i^\top e_j =
\begin{cases}
1, & \text{если } i = j, \\
0, & \text{если } i \neq j.
\end{cases}
$$

Данное равенство показывает, что one-hot векторы попарно ортогональны и имеют единичную норму. Именно эта ортогональность становится ключевым источником как достоинств (простота, однозначность), так и фатальных недостатков (отсутствие семантической близости), о которых пойдёт речь ниже.

## 1.4 Пример one-hot encoding на маленьком корпусе

Рассмотрим иллюстративный корпус, состоящий из трёх документов:

- Документ 1: «кошка сидит на окне»,
- Документ 2: «собака сидит на крыльце»,
- Документ 3: «кошка спит на диване».

После удаления знаков препинания и приведения к нормальной форме получаем следующий набор уникальных слов (словарь):

$$
V = \{ \text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване} \}.
$$

Размер словаря $N = 8$. Присвоим каждому слову индекс в порядке перечисления: кошка — 1, сидит — 2, на — 3, окне — 4, собака — 5, крыльце — 6, спит — 7, диване — 8. Тогда one-hot векторы для каждого слова будут иметь длину 8.

Для слова «кошка» (индекс 1) вектор равен:

$$
e_{\text{кошка}} = (1, 0, 0, 0, 0, 0, 0, 0)^\top .
$$

Для слова «сидит» (индекс 2):

$$
e_{\text{сидит}} = (0, 1, 0, 0, 0, 0, 0, 0)^\top .
$$

Аналогично,

$$
e_{\text{на}} = (0, 0, 1, 0, 0, 0, 0, 0)^\top ,
$$

$$
e_{\text{окне}} = (0, 0, 0, 1, 0, 0, 0, 0)^\top ,
$$

$$
e_{\text{собака}} = (0, 0, 0, 0, 1, 0, 0, 0)^\top ,
$$

$$
e_{\text{крыльце}} = (0, 0, 0, 0, 0, 1, 0, 0)^\top ,
$$

$$
e_{\text{спит}} = (0, 0, 0, 0, 0, 0, 1, 0)^\top ,
$$

$$
e_{\text{диване}} = (0, 0, 0, 0, 0, 0, 0, 1)^\top .
$$

Если мы захотим представить целый документ как последовательность one-hot векторов, то документ 1 будет записан как список из четырёх векторов:

$$
[ e_{\text{кошка}}, e_{\text{сидит}}, e_{\text{на}}, e_{\text{окне}} ].
$$

Такое представление сохраняет порядок слов, но каждый вектор имеет длину $N$, равную размеру словаря. Для нашего игрушечного корпуса $N=8$, что не вызывает проблем, однако при реальных объёмах словарь достигает сотен тысяч и миллионов, и хранение последовательностей таких векторов становится крайне неэффективным, как будет показано в следующем разделе.

## 1.5 Вычислительная и семантическая неэффективность one-hot encoding

### 1.5.1 Проблема высокой размерности и разреженности

Главным недостатком one-hot encoding является колоссальная размерность получаемых векторов. Если словарь содержит $N$ слов, то каждый вектор имеет длину $N$, а поскольку реальные $N$ могут составлять от $10^5$ до $10^7$, хранение плотного вектора для каждого вхождения слова становится непрактичным. Более того, подавляющее большинство компонентов вектора равны нулю. Доля ненулевых элементов равна $1/N$, что при $N = 10^6$ составляет $10^{-6}$, то есть 0,0001 %. Такие векторы называют разреженными (sparse). Хотя существуют специализированные форматы хранения разреженных матриц и векторов, они не устраняют фундаментальной проблемы: модель, работающая с такими представлениями, вынуждена оперировать пространством, в котором каждое слово изолировано от остальных.

Чтобы наглядно продемонстрировать масштаб проблемы, проведём мысленный эксперимент. Представьте небольшую библиотеку из 15 000 книг. Каждая книга содержит около 300 страниц, на каждой странице в среднем 35 строк, а в каждой строке — 20 слов. Тогда общее количество словоупотреблений во всей библиотеке равно:

$$
T = 15\,000 \times 300 \times 35 \times 20 = 3\,150\,000\,000 .
$$

Если словарь уникальных слов после нормализации составляет $N = 1\,000\,000$ (что реалистично для такого объёма), то каждый one-hot вектор займёт $N$ чисел. При использовании 32-битных чисел с плавающей точкой (float32), каждый вектор потребует $4 \times 10^6$ байт = 4 МБ. Для хранения векторов для всех вхождений потребуется $T \times 4 \times 10^6$ байт, то есть $12{,}6 \times 10^{15}$ байт, или 12,6 петабайт. Это на несколько порядков превышает объёмы, доступные даже крупным дата-центрам. Даже если использовать разреженное представление, храня только индекс ненулевого элемента, объём сократится до $T \times 4$ байт ≈ 12,6 ГБ, что уже приемлемо, но тогда мы фактически отказываемся от one-hot векторов в пользу простых индексов, и никакой дополнительной информации не получаем.

Таким образом, разреженное хранение one-hot векторов эквивалентно возврату к простой индексации слов, не дающей никакой дополнительной информации о их значениях. Поэтому one-hot encoding используется лишь как промежуточный технический приём, а не как самостоятельное представление для семантических задач.

Проблема не только в памяти, но и в вычислительной сложности. Матричные операции с такими векторами, например умножение матрицы весов, требуют либо огромных ресурсов, либо специальных разреженных процедур, которые всё равно ограничены. Но даже если бы память и вычисления были бесплатными, остаётся ещё более фундаментальная проблема — отсутствие семантической близости.

### 1.5.2 Отсутствие семантической близости

Из определения one-hot векторов следует, что скалярное произведение любых двух различных векторов равно нулю:

$$
e_i^\top e_j = 0 \quad \text{при } i \neq j .
$$

Это означает, что в данном векторном пространстве все слова одинаково далеки друг от друга с точки зрения евклидова расстояния или косинусной меры. Действительно, евклидово расстояние между $e_i$ и $e_j$ равно $\sqrt{2}$ для всех $i \neq j$, а косинусная близость равна 0. Следовательно, слова «кошка» и «собака» оказываются не ближе, чем «кошка» и «диван» или «кошка» и «на». Семантическая информация, присущая языку, полностью теряется. Дистрибутивная гипотеза, согласно которой слова со схожими значениями встречаются в схожих контекстах, в one-hot представлении не находит никакого отражения.

Более того, one-hot векторы не позволяют вводить понятие частичного сходства: невозможно сказать, что «кошка» и «котёнок» похожи, а «кошка» и «автомобиль» — нет. Каждое слово становится изолированной точкой на сфере в $N$-мерном пространстве, и никакая линейная или нелинейная модель, обученная на таких векторах без дополнительной информации, не сможет выявить семантические закономерности. Именно поэтому one-hot encoding используется в основном как промежуточный технический приём для кодирования категориальных признаков или как вход для embedding-слоёв нейронных сетей, где он немедленно преобразуется в плотные низкоразмерные векторы.

### 1.5.3 Проблема неограниченности словаря

Ещё один недостаток one-hot encoding связан с тем, что векторное пространство жёстко привязано к словарю обучающего корпуса. Если после обучения модели встречается слово, которого не было в корпусе (out-of-vocabulary, OOV), то его невозможно представить в том же векторном пространстве без перестройки всего представления. Приходится либо добавлять новое измерение и переобучать модель, либо заменять такие слова специальным токеном (например, `<unk>`), что приводит к потере информации. В реальных задачах OOV-слова возникают постоянно из-за новых терминов, имён, опечаток и морфологических вариаций, поэтому подобная негибкость является серьёзным ограничением.

Таким образом, one-hot encoding, будучи простым и интуитивно понятным способом представления, страдает от трёх фундаментальных проблем: высокая размерность и разреженность, отсутствие семантической близости, неспособность обрабатывать неизвестные слова. Эти ограничения мотивируют разработку более продвинутых методов векторизации, которые учитывают статистику совместной встречаемости слов и их контекстное распределение.

## 1.6 Роль one-hot encoding в современных архитектурах

Несмотря на перечисленные недостатки, one-hot encoding не исчез из практики. Он широко применяется в нейронных сетях как способ ввода категориальных данных. В частности, в моделях NLP на основе нейронных сетей каждое слово сначала преобразуется в one-hot вектор, который затем умножается на матрицу встраиваний (embedding matrix). Это умножение фактически выбирает строку матрицы, соответствующую индексу слова, и на выходе получается плотный низкоразмерный вектор. Именно эти плотные векторы, а не one-hot векторы, и используются далее в вычислениях. Таким образом, one-hot encoding служит лишь промежуточным звеном, обеспечивающим удобную индексацию, но не является финальным представлением.

Понимание one-hot encoding необходимо для осознания того, почему последующие методы — bag-of-words, TF-IDF, латентный семантический анализ, тематические модели и нейросетевые эмбеддинги — были разработаны и какие проблемы они призваны решить. Каждый из этих подходов добавляет всё больше семантической информации, сохраняя при этом вычислительную эффективность. В следующей части лекции мы перейдём к рассмотрению bag-of-words, который делает первый шаг от представления отдельного слова к представлению целого документа, хотя и ценой потери порядка слов.

# Часть 2. Bag of Words и TF-IDF: представление документов на основе частот

## 2.1 От представления слова к представлению документа

One-hot encoding, рассмотренное в предыдущей части, оперирует отдельными словами и не даёт никакого способа представить целый документ, кроме как в виде последовательности огромных разреженных векторов. Для многих задач — классификации текстов, кластеризации, поиска — необходимо иметь единый вектор фиксированной длины, который описывал бы весь документ. Наиболее естественный и исторически первый способ добиться этого — посчитать, сколько раз каждое слово из общего словаря встречается в данном документе, и записать эти частоты в вектор. Такой подход называется **Bag of Words**, или «мешок слов». Название отражает принципиальное допущение: порядок слов не учитывается, документ рассматривается как неупорядоченный набор слов, как будто все слова высыпали из предложения в мешок и перемешали.

Bag of Words решает проблему переменной длины документа: независимо от того, сколько слов в тексте, его представление всегда имеет размерность, равную размеру общего словаря $|V|$. Это позволяет использовать стандартные алгоритмы машинного обучения, работающие с векторами фиксированной длины. Кроме того, BoW впервые вводит идею, что слова, часто встречающиеся вместе в одних и тех же документах, могут указывать на тематическую близость документов. Например, если в двух документах часто встречаются слова «кошка», «сидит», «окно», то эти документы, вероятно, описывают похожие ситуации, даже если порядок слов различается.

## 2.2 Формальное определение Bag of Words

Пусть задана коллекция документов $D = \{d_1, d_2, \ldots, d_M\}$, где $M$ — количество документов. Из всех документов выделяется словарь $V$ — множество всех уникальных слов, встречающихся в коллекции, и его размер $N = |V|$. Занумеруем слова словаря: $V = \{w_1, w_2, \ldots, w_N\}$. Тогда каждый документ $d_m$ можно представить вектором $\mathbf{x}_m \in \mathbb{R}^N$, компоненты которого равны количеству вхождений соответствующего слова в документ:

$$
(\mathbf{x}_m)_j = \operatorname{tf}(w_j, d_m), \quad j = 1, \ldots, N,
$$

где $\operatorname{tf}(w_j, d_m)$ — частота слова $w_j$ в документе $d_m$, то есть число раз, которое слово $w_j$ встретилось в $d_m$. Такой вектор называется **вектором частот** или **Bag-of-Words вектором**.

Если составить матрицу $X \in \mathbb{R}^{M \times N}$, в которой $m$-я строка является вектором документа $d_m$, то получим **матрицу «документ-термин»**. Элемент $X_{mj}$ равен частоте слова $j$ в документе $m$. Эта матрица содержит всю информацию о частотах слов в коллекции, но не содержит информации о порядке слов внутри документов.

Важно отметить, что BoW-вектор может быть нормализован. Часто применяют L1-нормализацию (деление на сумму частот) или L2-нормализацию (деление на евклидову норму), чтобы уменьшить влияние длины документа. Если документ длинный, то в нём могут встречаться большие абсолютные частоты, что может неоправданно увеличивать его вес при сравнении с короткими документами. Нормализация приводит векторы к сопоставимому масштабу. Однако в базовом определении BoW используется именно сырая частота.

## 2.3 Пример Bag of Words на маленьком корпусе

Рассмотрим тот же учебный корпус, который использовался в части 1:

- Документ 1: «кошка сидит на окне»,
- Документ 2: «собака сидит на крыльце»,
- Документ 3: «кошка спит на диване».

Словарь этого корпуса состоит из восьми слов: $V = \{\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване}\}$, $N = 8$. Занумеруем слова по порядку: 1 — кошка, 2 — сидит, 3 — на, 4 — окне, 5 — собака, 6 — крыльце, 7 — спит, 8 — диване.

Подсчитаем частоты каждого слова в каждом документе.

Документ 1 содержит слова «кошка», «сидит», «на», «окне» каждое по одному разу. Следовательно, вектор документа 1:

$$
\mathbf{x}_1 = (1, 1, 1, 1, 0, 0, 0, 0)^\top.
$$

Документ 2: «собака», «сидит», «на», «крыльце» по одному разу:

$$
\mathbf{x}_2 = (0, 1, 1, 0, 1, 1, 0, 0)^\top.
$$

Документ 3: «кошка», «спит», «на», «диване» по одному разу:

$$
\mathbf{x}_3 = (1, 0, 1, 0, 0, 0, 1, 1)^\top.
$$

Матрица «документ-термин» $X$ размера $3 \times 8$ выглядит так:

$$
X = \begin{pmatrix}
1 & 1 & 1 & 1 & 0 & 0 & 0 & 0 \\
0 & 1 & 1 & 0 & 1 & 1 & 0 & 0 \\
1 & 0 & 1 & 0 & 0 & 0 & 1 & 1
\end{pmatrix}.
$$

Видно, что каждая строка соответствует документу, а каждый столбец — слову. Например, первый столбец (слово «кошка») имеет значения 1 в первой строке, 0 во второй, 1 в третьей. Это означает, что «кошка» встречается в документах 1 и 3, но не встречается в документе 2.

Если применить L1-нормализацию (каждый вектор разделить на сумму его компонентов, которая в нашем случае равна 4 для всех документов), то получим векторы:

$$
\mathbf{x}_1^{\text{norm}} = (0.25, 0.25, 0.25, 0.25, 0, 0, 0, 0),
$$
$$
\mathbf{x}_2^{\text{norm}} = (0, 0.25, 0.25, 0, 0.25, 0.25, 0, 0),
$$
$$
\mathbf{x}_3^{\text{norm}} = (0.25, 0, 0.25, 0, 0, 0, 0.25, 0.25).
$$

Такая нормализация делает векторы независимыми от длины документа, но в нашем примере длины совпадают.

## 2.4 Недостатки Bag of Words

Bag of Words страдает от нескольких принципиальных ограничений. Во-первых, полностью игнорируется порядок слов. Выражения «кошка сидит на собаке» и «собака сидит на кошке» дадут одинаковые BoW-векторы, хотя их смысл противоположен. Это не позволяет модели улавливать синтаксис и контекст. Во-вторых, частые служебные слова (например, «на», «и», «в») доминируют в векторах, хотя не несут основной смысловой нагрузки. В нашем примере слово «на» встречается во всех трёх документах и имеет высокую частоту, но оно не помогает различать документы по теме. В-третьих, размерность вектора по-прежнему равна размеру словаря, который в реальных задачах может достигать сотен тысяч, а сами векторы разрежены, что требует эффективных разреженных структур данных.

Кроме того, BoW не отражает семантическую близость слов. Слова «кошка» и «собака» в BoW представлены разными координатами, и их близость не выше, чем близость «кошка» и «диван». Для решения части этих проблем было предложено взвешивание частот слов с учётом их информативности — метод **TF-IDF**.

## 2.5 Идея TF-IDF

**TF-IDF** (Term Frequency – Inverse Document Frequency) модифицирует BoW, умножая частоту слова в документе на коэффициент, обратный частоте встречаемости слова во всей коллекции. Основная идея: слова, которые встречаются почти во всех документах (например, предлоги, союзы), несут мало информации для различения документов, поэтому их вес должен быть уменьшен. Напротив, слова, встречающиеся в небольшом числе документов, более специфичны и должны получать больший вес. Таким образом, TF-IDF усиливает значимость редких, но характерных терминов.

TF-IDF не является одной строгой формулой, а представляет семейство взвешивающих схем. В классическом варианте используются две компоненты: **TF** (частота в документе) и **IDF** (обратная документная частота), которые перемножаются.

## 2.6 Формальное определение TF-IDF

Для слова $w_j$ и документа $d_m$ определим:

- **Term Frequency** — частота слова в документе. Это может быть сырая частота $\operatorname{tf}(w_j, d_m)$, но на практике часто используют логарифмированную частоту, чтобы сгладить влияние многократных повторений одного слова. Например, $\operatorname{tf}_{\log}(w_j, d_m) = \log(1 + \operatorname{tf}(w_j, d_m))$.

- **Inverse Document Frequency** — мера редкости слова. Она вычисляется на основе количества документов, в которых слово встречается хотя бы один раз. Обозначим через $df(w_j)$ число документов, содержащих слово $w_j$. Тогда IDF определяется как:

$$
\operatorname{idf}(w_j) = \log \frac{M}{df(w_j)},
$$

где $M$ — общее число документов. Для предотвращения деления на ноль (если слово не встречается ни в одном документе, что невозможно для слов из словаря), иногда используют сглаживание: $\operatorname{idf}(w_j) = \log \frac{M}{1 + df(w_j)}$ или добавляют 1 к числителю и знаменателю. Мы будем использовать базовую формулу без сглаживания, поскольку все слова из словаря встречаются хотя бы один раз.

Тогда вес TF-IDF слова $w_j$ в документе $d_m$ равен произведению:

$$
\operatorname{tf\text{-}idf}(w_j, d_m) = \operatorname{tf}(w_j, d_m) \times \operatorname{idf}(w_j).
$$

Полный вектор документа в пространстве TF-IDF:

$$
\mathbf{x}_m^{\text{tfidf}} = \left( \operatorname{tf\text{-}idf}(w_1, d_m), \ldots, \operatorname{tf\text{-}idf}(w_N, d_m) \right).
$$

Если используется нормализация TF (например, лог-нормализация), то формула принимает вид $\operatorname{tf\text{-}idf}(w_j, d_m) = \operatorname{tf}_{\log}(w_j, d_m) \times \operatorname{idf}(w_j)$.

## 2.7 Пример TF-IDF на том же корпусе

Продолжим работу с нашим корпусом из трёх документов. Имеем $M = 3$. Вычислим IDF для каждого слова.

- Документ 1: «кошка сидит на окне»,
- Документ 2: «собака сидит на крыльце»,
- Документ 3: «кошка спит на диване».


Количество документов, содержащих слово:

- «кошка» встречается в документах 1 и 3, значит $df = 2$.
- «сидит» встречается в документах 1 и 2, $df = 2$.
- «на» встречается во всех трёх документах, $df = 3$.
- «окне» встречается только в документе 1, $df = 1$.
- «собака» только в документе 2, $df = 1$.
- «крыльце» только в документе 2, $df = 1$.
- «спит» только в документе 3, $df = 1$.
- «диване» только в документе 3, $df = 1$.

Теперь вычислим IDF по формуле $\log(M / df)$:

- Для «кошка»: $\log(3/2) \approx 0.405$.
- Для «сидит»: $\log(3/2) \approx 0.405$.
- Для «на»: $\log(3/3) = 0$.
- Для «окне»: $\log(3/1) \approx 1.099$.
- Для «собака»: $\log(3/1) \approx 1.099$.
- Для «крыльце»: $\log(3/1) \approx 1.099$.
- Для «спит»: $\log(3/1) \approx 1.099$.
- Для «диване»: $\log(3/1) \approx 1.099$.

Заметим, что слово «на» получило нулевой IDF, поскольку встречается во всех документах и не помогает их различать. Это решает проблему доминирования частых слов.

Теперь вычислим TF-IDF для каждого документа, используя сырую частоту (все частоты равны 1 для встречающихся слов). Для документа 1:

- кошка: $1 \times 0.405 = 0.405$
- сидит: $1 \times 0.405 = 0.405$
- на: $1 \times 0 = 0$
- окне: $1 \times 1.099 = 1.099$
- остальные: 0.

Вектор документа 1:

$$
\mathbf{x}_1^{\text{tfidf}} = (0.405, 0.405, 0, 1.099, 0, 0, 0, 0).
$$

Документ 2:

- собака: $1 \times 1.099 = 1.099$
- сидит: $0.405$
- на: $0$
- крыльце: $1.099$

$$
\mathbf{x}_2^{\text{tfidf}} = (0, 0.405, 0, 0, 1.099, 1.099, 0, 0).
$$

Документ 3:

- кошка: $0.405$
- спит: $1.099$
- на: $0$
- диване: $1.099$

$$
\mathbf{x}_3^{\text{tfidf}} = (0.405, 0, 0, 0, 0, 0, 1.099, 1.099).
$$

Матрица TF-IDF:

$$
X_{\text{tfidf}} = \begin{pmatrix}
0.405 & 0.405 & 0 & 1.099 & 0 & 0 & 0 & 0 \\
0 & 0.405 & 0 & 0 & 1.099 & 1.099 & 0 & 0 \\
0.405 & 0 & 0 & 0 & 0 & 0 & 1.099 & 1.099
\end{pmatrix}.
$$

Видно, что служебное слово «на» обнулилось, а слова, характерные для одного документа, получили наибольший вес. Это улучшает различение документов.

Если использовать лог-нормализацию TF, $\operatorname{tf}_{\log} = \log(1+1) = \log 2 \approx 0.693$, то веса изменятся: «окне» стало бы $0.693 \times 1.099 \approx 0.762$, но пропорции сохранятся.

## 2.8 Свойства и ограничения TF-IDF

TF-IDF является де-факто стандартом для многих задач информационного поиска и классификации текстов. Он эффективно снижает влияние частых слов и подчёркивает важные термины. Однако он сохраняет основные ограничения BoW: порядок слов игнорируется, семантическая синонимия не учитывается, векторы остаются разреженными и высокоразмерными. Более того, TF-IDF не способен уловить скрытые тематические структуры, которые могут объединять слова, даже если они не встречаются в одних и тех же документах напрямую.

Для преодоления этих ограничений были разработаны методы, основанные на матричной факторизации и вероятностных тематических моделях, о которых пойдёт речь в следующей части. Тем не менее, BoW и TF-IDF остаются важными базовыми инструментами, и понимание их математики необходимо для освоения более сложных подходов.

# Часть 3. Матричные методы на основе совместной встречаемости: HAL и LSA

## 3.1 От частот к контекстам

Представления Bag of Words и TF-IDF, рассмотренные в предыдущей части, основаны на частотах слов в документах и не учитывают связи между словами. Они не позволяют выявить семантическую близость: слова «кошка» и «собака» оказываются такими же далёкими друг от друга, как «кошка» и «диван», хотя интуитивно мы понимаем, что первые два слова семантически связаны. Причина в том, что эти методы представляют каждое слово как отдельную координату, независимую от остальных. Для того чтобы уловить смысловое сходство, необходимо опереться на **дистрибутивную гипотезу**, сформулированную лингвистами ещё в середине XX века: слова, встречающиеся в похожих контекстах, имеют похожие значения. Иными словами, значение слова определяется его окружением.

Эта идея приводит к построению векторных представлений, основанных на **совместной встречаемости** слов в пределах некоторого окна. Если два слова часто появляются рядом с одними и теми же другими словами, их векторы должны быть похожи. Матричные методы, такие как **HAL** (Hyperspace Analogue to Language) и **LSA** (Latent Semantic Analysis), используют этот принцип, строя матрицы совместной встречаемости или матрицы «термин-документ», а затем применяя к ним линейную алгебру для получения плотных векторов слов или документов.

В этой части мы рассмотрим два классических метода: HAL, который строит векторы слов напрямую из матрицы совместной встречаемости без дополнительной факторизации, и LSA, который применяет сингулярное разложение (SVD) к матрице «термин-документ», чтобы выявить латентные семантические факторы.


## 3.2 HAL: Hyperspace Analogue to Language

### 3.2.1 Идея и построение матрицы совместной встречаемости

HAL был предложен Лундом и Бёрджессом в 1996 году как модель семантической памяти, имитирующая то, как человек усваивает значения слов из опыта чтения. Основная идея заключается в том, чтобы для каждого слова в корпусе собрать информацию о его соседях в скользящем окне. Чем чаще два слова встречаются рядом, тем сильнее их семантическая связь. В результате каждое слово получает вектор, компонентами которого являются меры совместной встречаемости с каждым другим словом словаря.

*В этом разделе мы сначала опишем упрощённую симметричную версию HAL, а затем укажем на особенности оригинальной модели, которая различает левый и правый контексты. Это различие важно для точного понимания метода, хотя в учебных примерах для простоты часто используют симметричный вариант.*

Формально, пусть у нас есть корпус, представленный последовательностью слов $w_1, w_2, \ldots, w_T$, где $T$ — общее число словоупотреблений. Зафиксируем размер окна $m$ (например, $m = 1$ или $m = 10$). Для каждой позиции $t$ мы рассматриваем пары $(w_t, w_{t+j})$, где $j \in \{-m, \ldots, -1, 1, \ldots, m\}$. Каждой такой паре приписывается вес, зависящий от расстояния $|j|$: чем ближе слово, тем больше вес. В оригинальной модели HAL используется линейное убывание веса с расстоянием, например $\frac{1}{|j|}$, но для простоты часто берут постоянный вес $1$.

Строится матрица совместной встречаемости $H \in \mathbb{R}^{N \times N}$, где $N = |V|$ — размер словаря. Элемент $H_{ij}$ равен суммарному весу всех вхождений пары $(w_i, w_j)$, где $w_i$ — целевое слово (в позиции $t$), а $w_j$ — контекстное слово (в позиции $t+j$).

*В упрощённой симметричной версии, которую мы будем использовать в примере, мы не различаем левый и правый контекст: $H_{ij}$ учитывает все появления слова $w_j$ в окне вокруг $w_i$ независимо от направления. Тогда матрица $H$ симметрична, так как $H_{ij} = H_{ji}$. Это удобно для первого знакомства с методом, однако оригинальная модель HAL **не является симметричной**.*

*В оригинальной модели HAL информация о левых и правых соседях обрабатывается раздельно. Для каждого слова $w_i$ строится два вектора: один для контекста слева (слова, стоящие до $w_i$ в пределах окна) и один для контекста справа (слова после $w_i$). Эти два вектора затем конкатенируются, образуя итоговый вектор размерности $2N$. Такой подход позволяет сохранить информацию о порядке слов относительно целевого слова, что важно для синтаксических и семантических нюансов.*

*Далее в учебных целях мы будем применять упрощённый симметричный вариант HAL, поскольку он достаточен для иллюстрации базовой идеи и не требует удвоения размерности. При использовании оригинальной несимметричной модели все рассуждения легко обобщаются заменой матрицы $H$ на пару матриц $H^{\text{left}}$ и $H^{\text{right}}$ с последующей конкатенацией их строк.*

Вектором слова $w_i$ в HAL обычно является строка матрицы $H$, то есть вектор $h_i = (H_{i1}, H_{i2}, \ldots, H_{iN})$. Этот вектор показывает, с какими словами и насколько часто встречалось слово $w_i$ в контексте. Часто строки нормализуют (например, L2-норма), чтобы компенсировать разную частоту слов.






## 3.2.2 Пример HAL на нашем корпусе — подробный разбор построения матрицы

### Что такое матрица совместной встречаемости?

Матрица совместной встречаемости $H$ — это квадратная таблица размером $N \times N$, где $N$ — размер словаря. Каждая её строка и каждый столбец соответствуют одному слову из словаря. Элемент $H_{ij}$ показывает, **сколько раз слово $j$ встречалось в окне рядом со словом $i$** (в симметричной версии — сколько раз слова $i$ и $j$ были соседями в пределах заданного окна).

В нашем примере:
- Словарь $V = \{\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване}\}$;
- $N = 8$;
- Нумерация: 1 — кошка, 2 — сидит, 3 — на, 4 — окне, 5 — собака, 6 — крыльце, 7 — спит, 8 — диване.

Размер окна $m=1$, то есть мы рассматриваем только **непосредственных соседей** слева и справа от каждого слова. Вес каждой соседней пары равен 1. Мы **не различаем направление**: если слово A стоит рядом со словом B (слева или справа), мы увеличиваем и $H_{AB}$, и $H_{BA}$ на 1. В результате матрица получается симметричной.

### Исходная последовательность слов

Объединим все документы в одну последовательность (границы документов игнорируем):

$$
\text{кошка}_1 \quad \text{сидит} \quad \text{на} \quad \text{окне} \quad \text{собака} \quad \text{сидит} \quad \text{на} \quad \text{крыльце} \quad \text{кошка}_2 \quad \text{спит} \quad \text{на} \quad \text{диване}
$$

Индексы $_1$ и $_2$ здесь используются только для того, чтобы различать два вхождения слова «кошка»; в матрице оба соответствуют одному индексу 1.

### Построение матрицы по шагам

Проходим по последовательности слева направо. Для каждого слова смотрим на его соседей (если они есть) и увеличиваем соответствующие элементы матрицы.

#### Первое вхождение «кошка» (позиция 1)

- Слева ничего нет.
- Справа — «сидит» (индекс 2).  
  Увеличиваем $H_{1,2}$ на 1 и симметрично $H_{2,1}$ на 1.

После шага: $H_{1,2}=1$, $H_{2,1}=1$, остальные — 0.

#### Слово «сидит» (первое вхождение, позиция 2)

- Слева — «кошка» (уже учтено).
- Справа — «на» (индекс 3).  
  Увеличиваем $H_{2,3}$ на 1 и $H_{3,2}$ на 1.

#### Слово «на» (первое вхождение, позиция 3)

- Слева — «сидит» (учтено).
- Справа — «окне» (индекс 4).  
  Увеличиваем $H_{3,4}$ на 1 и $H_{4,3}$ на 1.

#### Слово «окне» (позиция 4)

- Слева — «на» (учтено).
- Справа — «собака» (индекс 5).  
  Увеличиваем $H_{4,5}$ на 1 и $H_{5,4}$ на 1.

#### Слово «собака» (позиция 5)

- Слева — «окне» (учтено).
- Справа — «сидит» (индекс 2).  
  Увеличиваем $H_{5,2}$ на 1 и $H_{2,5}$ на 1.

Теперь у слова «сидит» (индекс 2) есть два соседства: с «кошка» ($H_{2,1}=1$) и с «собака» ($H_{2,5}=1$).

#### Слово «сидит» (второе вхождение, позиция 6)

- Слева — «собака» (уже учтено на предыдущем шаге, повторно не увеличиваем).
- Справа — «на» (индекс 3).  
  Пара «сидит»–«на» встречается уже второй раз, поэтому увеличиваем $H_{2,3}$ и $H_{3,2}$ ещё на 1.  
  Теперь $H_{2,3}=2$, $H_{3,2}=2$.

#### Слово «на» (второе вхождение, позиция 7)

- Слева — «сидит» (учтено, $H_{3,2}=2$).
- Справа — «крыльце» (индекс 6).  
  Увеличиваем $H_{3,6}$ на 1 и $H_{6,3}$ на 1.

#### Слово «крыльце» (позиция 8)

- Слева — «на» (учтено).
- Справа — «кошка» (индекс 1).  
  Увеличиваем $H_{6,1}$ на 1 и $H_{1,6}$ на 1.  
  Это соседство относится ко второму вхождению «кошки».

#### Второе вхождение «кошка» (позиция 9)

- Слева — «крыльце» (уже учтено, $H_{1,6}=1$).
- Справа — «спит» (индекс 7).  
  Увеличиваем $H_{1,7}$ на 1 и $H_{7,1}$ на 1.

Теперь у «кошки» есть три соседства: с «сидит» (1 раз), с «крыльце» (1 раз), с «спит» (1 раз).

#### Слово «спит» (позиция 10)

- Слева — «кошка» (учтено).
- Справа — «на» (индекс 3).  
  Увеличиваем $H_{7,3}$ на 1 и $H_{3,7}$ на 1.

#### Слово «на» (третье вхождение, позиция 11)

- Слева — «спит» (учтено).
- Справа — «диване» (индекс 8).  
  Увеличиваем $H_{3,8}$ на 1 и $H_{8,3}$ на 1.

#### Слово «диване» (позиция 12)

- Слева — «на» (учтено).
- Справа соседей нет.

### Итоговая матрица $H$

После обработки всех позиций получаем симметричную матрицу $8 \times 8$:

$$
\begin{array}{c|cccccccc}
 & 1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 \\
\hline
1 & 0 & 1 & 0 & 0 & 0 & 1 & 1 & 0 \\
2 & 1 & 0 & 2 & 0 & 1 & 0 & 0 & 0 \\
3 & 0 & 2 & 0 & 1 & 0 & 1 & 1 & 1 \\
4 & 0 & 0 & 1 & 0 & 1 & 0 & 0 & 0 \\
5 & 0 & 1 & 0 & 1 & 0 & 0 & 0 & 0 \\
6 & 1 & 0 & 1 & 0 & 0 & 0 & 0 & 0 \\
7 & 1 & 0 & 1 & 0 & 0 & 0 & 0 & 0 \\
8 & 0 & 0 & 1 & 0 & 0 & 0 & 0 & 0
\end{array}
$$

Или в более привычном матричном виде:

$$
H = \begin{pmatrix}
0 & 1 & 0 & 0 & 0 & 1 & 1 & 0 \\
1 & 0 & 2 & 0 & 1 & 0 & 0 & 0 \\
0 & 2 & 0 & 1 & 0 & 1 & 1 & 1 \\
0 & 0 & 1 & 0 & 1 & 0 & 0 & 0 \\
0 & 1 & 0 & 1 & 0 & 0 & 0 & 0 \\
1 & 0 & 1 & 0 & 0 & 0 & 0 & 0 \\
1 & 0 & 1 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 1 & 0 & 0 & 0 & 0 & 0
\end{pmatrix}
$$

Матрица симметрична: $H_{ij} = H_{ji}$. Например, $H_{1,2} = H_{2,1} = 1$, $H_{2,3} = H_{3,2} = 2$.

### Что означают строки матрицы?

Каждая строка матрицы — это вектор соответствующего слова.

#### Вектор слова «кошка» (индекс 1)

$$
v_{\text{кошка}} = (0,\ 1,\ 0,\ 0,\ 0,\ 1,\ 1,\ 0)
$$

- $H_{1,2}=1$ — слово «сидит» было соседом «кошки» 1 раз (в самом начале: «кошка сидит»).
- $H_{1,6}=1$ — слово «крыльце» было соседом 1 раз («крыльце кошка»).
- $H_{1,7}=1$ — слово «спит» было соседом 1 раз («кошка спит»).
- Остальные позиции равны нулю.

#### Вектор слова «сидит» (индекс 2)

$$
v_{\text{сидит}} = (1,\ 0,\ 2,\ 0,\ 1,\ 0,\ 0,\ 0)
$$

- $H_{2,1}=1$ — соседство с «кошка».
- $H_{2,3}=2$ — соседство с «на» встречалось дважды («сидит на окне» и «сидит на крыльце»).
- $H_{2,5}=1$ — соседство с «собака» («собака сидит»).

#### Вектор слова «на» (индекс 3)

$$
v_{\text{на}} = (0,\ 2,\ 0,\ 1,\ 0,\ 1,\ 1,\ 1)
$$

Слово «на» часто выступает связкой, поэтому имеет много ненулевых связей: с «сидит» (2 раза), с «окне», «крыльце», «спит», «диване» (по 1 разу).

### Сравнение векторов слов

Теперь, имея векторы, можно вычислять косинусное сходство, чтобы оценить семантическую близость слов, основанную на их контекстах.

Возьмём слова «кошка» и «собака»:

$$
v_{\text{кошка}} = (0,1,0,0,0,1,1,0)
$$
$$
v_{\text{собака}} = (0,1,0,1,0,0,0,0)
$$

Скалярное произведение:

$$
v_{\text{кошка}} \cdot v_{\text{собака}} = 0\cdot0 + 1\cdot1 + 0\cdot0 + 0\cdot1 + 0\cdot0 + 1\cdot0 + 1\cdot0 + 0\cdot0 = 1
$$

Нормы векторов:

$$
\|v_{\text{кошка}}\| = \sqrt{0^2 + 1^2 + 0^2 + 0^2 + 0^2 + 1^2 + 1^2 + 0^2} = \sqrt{3}
$$
$$
\|v_{\text{собака}}\| = \sqrt{0^2 + 1^2 + 0^2 + 1^2 + 0^2 + 0^2 + 0^2 + 0^2} = \sqrt{2}
$$

Косинусное сходство:

$$
\cos(\theta) = \frac{1}{\sqrt{3}\cdot\sqrt{2}} = \frac{1}{\sqrt{6}} \approx 0.408
$$

Ненулевое значение ($0.408$) указывает на то, что слова «кошка» и «собака» имеют некоторое сходство, поскольку оба встречаются рядом со словом «сидит». В one-hot представлении это сходство было бы строго равно нулю.

### Замечания по реализации

- В реальной модели HAL окно обычно больше (например, $m = 4$–$10$), а вес соседства убывает с расстоянием (например, $\frac{1}{|j|}$). Это делает векторы более гладкими и содержательными.
- Часто перед построением матрицы применяют взвешивание, аналогичное TF-IDF, чтобы снизить влияние слишком частых слов (например, предлога «на», у которого много связей).
- В оригинальной модели HAL левый и правый контексты хранятся раздельно и затем конкатенируются; в нашем учебном примере мы использовали симметричную версию для простоты.

Таким образом, матрица совместной встречаемости $H$ является центральным элементом метода HAL: её строки служат векторными представлениями слов, отражающими их контекстную дистрибуцию, что позволяет улавливать семантическую близость.


### 3.2.3 Ограничения HAL

HAL страдает от высокой размерности (равной размеру словаря) и разреженности, поскольку большинство пар слов никогда не встречаются в пределах окна. Кроме того, он чувствителен к частотным словам: если не применять нормализацию, частые слова будут иметь большие значения. Для устранения этих проблем часто применяют взвешивание, аналогичное TF-IDF, или методы понижения размерности, такие как SVD, что приводит нас к LSA.



## 3.3 LSA: Latent Semantic Analysis

### 3.3.1 Мотивация и идея

LSA, предложенный в 1990 году, решает проблему разреженности и высокой размерности, а также частично проблему синонимии, путём применения сингулярного разложения к матрице «термин-документ». Основная идея состоит в том, что между словами и документами существует скрытая (латентная) семантическая структура, которая может быть выявлена с помощью линейной алгебры. LSA проецирует слова и документы в низкоразмерное пространство, где семантически близкие объекты оказываются рядом.

В отличие от HAL, который строит матрицу совместной встречаемости слов, LSA обычно работает с матрицей частот слов в документах (часто взвешенной TF-IDF). Каждая строка этой матрицы соответствует слову, а каждый столбец — документу, либо наоборот. Применяя SVD, мы получаем разложение, которое позволяет аппроксимировать исходную матрицу произведением трёх матриц меньшего ранга. Это разложение выделяет главные направления вариации данных, которые интерпретируются как латентные темы или факторы.

### 3.3.2 Математическая основа SVD

Пусть у нас есть матрица $X \in \mathbb{R}^{m \times n}$, где $m$ — количество строк (например, слов), $n$ — количество столбцов (например, документов). Сингулярное разложение матрицы $X$ имеет вид:

$$
X = U \Sigma V^\top,
$$

где:

- $U \in \mathbb{R}^{m \times m}$ — ортонормированная матрица левых сингулярных векторов, столбцы которой являются собственными векторами матрицы $XX^\top$;
- $\Sigma \in \mathbb{R}^{m \times n}$ — диагональная матрица, на главной диагонали которой стоят сингулярные числа $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r > 0$, где $r = \operatorname{rank}(X)$;
- $V \in \mathbb{R}^{n \times n}$ — ортонормированная матрица правых сингулярных векторов, столбцы которой являются собственными векторами матрицы $X^\top X$.

Если взять только $k$ наибольших сингулярных чисел и соответствующие столбцы $U$ и $V$, то получим **усечённое SVD**:

$$
X_k = U_k \Sigma_k V_k^\top,
$$

где $U_k \in \mathbb{R}^{m \times k}$, $\Sigma_k \in \mathbb{R}^{k \times k}$, $V_k \in \mathbb{R}^{n \times k}$. Матрица $X_k$ является наилучшим приближением матрицы $X$ ранга $k$ в смысле как спектральной нормы, так и нормы Фробениуса (теорема Эккарта–Янга). Это означает, что мы можем существенно снизить размерность, сохранив основную структуру данных.

В контексте LSA обычно строят матрицу $X$ размера $N \times M$ (слова × документы) или $M \times N$ (документы × слова). Для определённости будем считать, что строки — слова, столбцы — документы. Тогда:

- Векторы слов в латентном пространстве получаются как строки матрицы $U_k$ (или $U_k \Sigma_k$). Каждая строка соответствует слову и имеет длину $k$.
- Векторы документов получаются как строки матрицы $V_k$ (или $V_k \Sigma_k$). Каждая строка соответствует документу.

Для сравнения слов или документов используют косинусную близость в $k$-мерном пространстве. Часто используют нормализованные векторы $U_k$ и $V_k$, так как сингулярные числа уже учтены в масштабе.

*На практике предпочтительнее использовать \(U_k \Sigma_k\) для векторов слов и \(V_k \Sigma_k\) для векторов документов, поскольку эти матрицы явно включают масштаб сингулярных чисел, отражающий важность каждой латентной размерности. Если же требуется сравнивать слова и документы в одном пространстве (например, запрос как вектор слов против документов), можно использовать \(U_k\) и \(V_k\), но тогда необходимо помнить, что масштаб сингулярных чисел не учтён. В учебных примерах для простоты допустимо работать с \(U_k\) и \(V_k\), если \(k\) невелико и относительные расстояния важнее абсолютных значений.*

### 3.3.3 Пример LSA на нашем корпусе

Используем матрицу TF-IDF из части 2 для нашего корпуса. Напомним, что после TF-IDF мы получили матрицу $X$ размера $8 \times 3$ (слова × документы), где строки соответствуют словам в порядке: кошка, сидит, на, окне, собака, крыльце, спит, диване, а столбцы — документам 1, 2, 3. Матрица имеет вид (округлённо до трёх знаков):

$$
X = \begin{pmatrix}
0.405 & 0 & 0.405 \\
0.405 & 0.405 & 0 \\
0 & 0 & 0 \\
1.099 & 0 & 0 \\
0 & 1.099 & 0 \\
0 & 1.099 & 0 \\
0 & 0 & 1.099 \\
0 & 0 & 1.099
\end{pmatrix}.
$$

Применим SVD к этой матрице. Для ручного вычисления SVD матрицы $8 \times 3$ достаточно трудоёмко, поэтому приведём условные, но правдоподобные результаты, иллюстрирующие идею. Предположим, что мы выбрали $k=2$ латентных измерения. После вычислений получим матрицы $U_2$, $\Sigma_2$, $V_2$ (округлённо):

$$
\Sigma_2 = \begin{pmatrix}
2.0 & 0 \\
0 & 1.5
\end{pmatrix},
$$

$$
U_2 = \begin{pmatrix}
-0.35 & 0.40 \\
-0.30 & -0.50 \\
-0.05 & 0.10 \\
-0.40 & 0.30 \\
-0.35 & -0.40 \\
-0.35 & -0.40 \\
-0.40 & 0.30 \\
-0.40 & 0.30
\end{pmatrix}.
$$

$$
V_2 = \begin{pmatrix}
-0.45 & -0.55 \\
-0.50 & 0.60 \\
-0.55 & -0.35
\end{pmatrix}.
$$

*Для простоты иллюстрации мы будем анализировать непосредственно строки матриц \(U_2\) и \(V_2\), не умножая их на \(\Sigma_2\). В реальных приложениях умножение на \(\Sigma_2\) может изменить относительные масштабы, но для нашего учебного примера это не влияет на качественные выводы о взаимном расположении слов и документов.*

Тогда векторы слов (строки $U_2$) в двумерном пространстве:

- кошка: $(-0.35, 0.40)$
- сидит: $(-0.30, -0.50)$
- на: $(-0.05, 0.10)$
- окне: $(-0.40, 0.30)$
- собака: $(-0.35, -0.40)$
- крыльце: $(-0.35, -0.40)$
- спит: $(-0.40, 0.30)$
- диване: $(-0.40, 0.30)$

Векторы документов (строки $V_2$, можно умножить на $\Sigma_2$, но для сравнения часто используют просто $V_2$):

- Документ 1: $(-0.45, -0.55)$
- Документ 2: $(-0.50, 0.60)$
- Документ 3: $(-0.55, -0.35)$

Интерпретируем полученные результаты. Слово «на» получило вектор близкий к нулю по первой координате и небольшой по второй, что отражает его низкую информативность (IDF=0). Слова «окне», «спит», «диване» имеют одинаковый вектор $(-0.40, 0.30)$, что указывает на то, что они связаны с одной латентной темой (возможно, «действия и места, характерные для кошки»). Слова «собака» и «крыльце» имеют вектор $(-0.35, -0.40)$, группируясь во вторую тему. Слово «кошка» занимает промежуточное положение между этими группами, так как встречается в документах с разными темами. Документы 1 и 3 расположены ближе друг к другу (оба про кошку), чем к документу 2 (про собаку). Это согласуется с интуицией.

Если вычислить косинусную близость между векторами слов «кошка» и «собака», она будет положительной (оба имеют отрицательную первую координату и разные знаки второй), в отличие от нулевой близости в one-hot. Таким образом, LSA улавливает семантическое сходство, основанное на совместной встречаемости в документах.

### 3.3.4 Свойства и ограничения LSA

LSA имеет ряд достоинств: она снижает размерность, устраняет шум, улавливает латентные семантические связи и позволяет сравнивать слова и документы в общем пространстве. Однако у неё есть и ограничения. Во-первых, SVD требует значительных вычислительных ресурсов для больших матриц, хотя существуют эффективные разреженные алгоритмы. *Полное сингулярное разложение матрицы размера \(m \times n\) имеет временную сложность \(O(mn \cdot \min(m,n))\), что для корпусов с сотнями тысяч слов и миллионов документов становится практически неприемлемым. Именно это ограничение стимулировало разработку более масштабируемых вероятностных тематических моделей, таких как pLSA и LDA.* Во-вторых, выбор числа измерений $k$ эвристичен и сильно влияет на результат. В-третьих, LSA не является вероятностной моделью: она не описывает процесс порождения текста и не имеет чёткой статистической интерпретации. Кроме того, LSA плохо работает с полисемией: слово с несколькими значениями усредняется в одном векторе.

Эти ограничения мотивировали разработку вероятностных тематических моделей, таких как pLSA и LDA, которые явно вводят скрытые темы и распределения вероятностей, что будет рассмотрено в следующей части.


# Часть 4.1. Вероятностный латентный семантический анализ (pLSA)

## 4.1.1 От линейной алгебры к вероятностной модели

Латентный семантический анализ, рассмотренный в предыдущей части, позволил снизить размерность и выявить скрытые факторы, однако он остаётся чисто алгебраическим методом. LSA не задаёт вероятностного процесса порождения текста, не даёт естественного способа обрабатывать новые документы и не имеет статистической интерпретации латентных измерений. Эти ограничения мотивировали разработку вероятностных тематических моделей, в которых документ рассматривается как смесь скрытых тем, а каждая тема — как распределение вероятностей над словами. Такой подход позволяет формулировать задачу в терминах максимума правдоподобия, применять стандартные методы статистического вывода и получать интерпретируемые результаты.

Первой и наиболее простой вероятностной тематической моделью является **вероятностный латентный семантический анализ** (probabilistic Latent Semantic Analysis, pLSA), предложенный Хофманом в 1999 году. pLSA вводит скрытую переменную — тему — и описывает порождение каждого слова в документе как двухступенчатый процесс: сначала выбирается тема согласно распределению тем документа, затем из выбранной темы генерируется слово. В отличие от LSA, pLSA имеет явную вероятностную интерпретацию и обучается методом максимального правдоподобия.

## 4.1.2 Порождающая модель и параметры

Пусть коллекция состоит из $M$ документов, а словарь содержит $N$ уникальных слов. Введём скрытую переменную $z \in \{1, 2, \dots, K\}$, где $K$ — заданное число тем. Каждый документ $d$ характеризуется распределением вероятностей тем $P(z \mid d)$, а каждая тема $z$ — распределением вероятностей слов $P(w \mid z)$. Предполагается, что порождение документа происходит следующим образом: для каждой позиции слова в документе сначала выбирается тема $z$ согласно $P(z \mid d)$, а затем из выбранной темы генерируется слово $w$ согласно $P(w \mid z)$.

Вероятность появления слова $w$ в документе $d$ записывается как

$$
P(w, d) = P(d) \sum_{z=1}^{K} P(w \mid z) P(z \mid d),
$$

где $P(d)$ — априорная вероятность документа, которую обычно не параметризуют и считают заданной эмпирической частотой (например, $P(d) = 1/M$). Параметрами модели являются два набора условных распределений:

- $\theta_{dz} = P(z \mid d)$ — распределение тем в документе $d$;
- $\phi_{zw} = P(w \mid z)$ — распределение слов в теме $z$.

Общее число параметров равно $M \times K + K \times N$. При $K \ll \min(M,N)$ это значительно меньше, чем размерность матрицы «термин-документ», но всё же растёт линейно с числом документов, что является одним из недостатков pLSA.

## 4.1.3 Функция правдоподобия

Пусть $n(d,w)$ — частота слова $w$ в документе $d$ (в нашем примере $n(d,w)$ равна либо 0, либо 1, так как каждое слово встречается не более одного раза). Полное правдоподобие коллекции:

$$
\mathcal{L} = \prod_{d=1}^{M} \prod_{w \in V} P(w, d)^{n(d,w)}.
$$

Логарифм правдоподобия (с точностью до слагаемого, не зависящего от параметров, так как $P(d)$ фиксирована):

$$
\ell = \sum_{d=1}^{M} \sum_{w \in V} n(d,w) \log \left( \sum_{z=1}^{K} P(w \mid z) P(z \mid d) \right).
$$

Цель — найти такие $\{\phi_{zw}\}$ и $\{\theta_{dz}\}$, которые максимизируют $\ell$ при ограничениях $\sum_w \phi_{zw} = 1$ и $\sum_z \theta_{dz} = 1$.

## 4.1.4 EM-алгоритм для pLSA

Для максимизации логарифма правдоподобия применяется **EM-алгоритм** (Expectation-Maximization). Он состоит из двух шагов, повторяемых до сходимости.

### E-шаг

Для каждой пары $(d,w)$ с $n(d,w) > 0$ вычисляется апостериорная вероятность темы $z$ при условии, что в документе $d$ встретилось слово $w$:

$$
P(z \mid d, w) = \frac{P(w \mid z) P(z \mid d)}{\sum_{z'=1}^{K} P(w \mid z') P(z' \mid d)}.
$$

Эта формула следует из правила Байеса, поскольку $P(w \mid z)$ и $P(z \mid d)$ известны на текущей итерации.

### M-шаг

Параметры обновляются, используя вычисленные апостериорные вероятности. Новые оценки:

$$
P(w \mid z) = \frac{\sum_{d=1}^{M} n(d,w) P(z \mid d, w)}{\sum_{d=1}^{M} \sum_{w' \in V} n(d,w') P(z \mid d, w')},
$$

$$
P(z \mid d) = \frac{\sum_{w \in V} n(d,w) P(z \mid d, w)}{\sum_{w \in V} n(d,w)}.
$$

Знаменатель во второй формуле равен длине документа $N_d$ (числу словоупотреблений в документе $d$). Таким образом, $P(z \mid d)$ — это среднее апостериорных вероятностей темы по всем словам документа.

EM гарантирует неубывание логарифма правдоподобия на каждой итерации, но сходится к локальному максимуму, поэтому начальная инициализация может влиять на результат.



### 4.1.5 Численный пример pLSA на учебном корпусе

Рассмотрим тот же корпус из трёх документов:

- Документ 1 ($d_1$): «кошка сидит на окне»;
- Документ 2 ($d_2$): «собака сидит на крыльце»;
- Документ 3 ($d_3$): «кошка спит на диване».

Словарь: $V = \{\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване}\}$, $N=8$.  
Число документов $M=3$, число тем $K=2$.

#### Инициализация параметров

Зададим несимметричные начальные распределения.

**Тема 1 ($z=1$):**

| слово | $P(w \mid z=1)$ |
|-------|-----------------|
| кошка | 0.25 |
| сидит | 0.20 |
| на    | 0.08 |
| окне  | 0.03 |
| собака | 0.25 |
| крыльце | 0.02 |
| спит  | 0.15 |
| диване | 0.02 |

Сумма: $0.25+0.20+0.08+0.03+0.25+0.02+0.15+0.02 = 1.00$.

**Тема 2 ($z=2$):**

| слово | $P(w \mid z=2)$ |
|-------|-----------------|
| кошка | 0.03 |
| сидит | 0.05 |
| на    | 0.30 |
| окне  | 0.25 |
| собака | 0.01 |
| крыльце | 0.20 |
| спит  | 0.01 |
| диване | 0.15 |

Сумма: $0.03+0.05+0.30+0.25+0.01+0.20+0.01+0.15 = 1.00$.

**Распределения тем по документам:**

- $P(z=1 \mid d_1) = 0.6$, $P(z=2 \mid d_1) = 0.4$;
- $P(z=1 \mid d_2) = 0.3$, $P(z=2 \mid d_2) = 0.7$;
- $P(z=1 \mid d_3) = 0.55$, $P(z=2 \mid d_3) = 0.45$.

#### Итерация 1: E-шаг

Для каждой пары (документ, слово), где слово встречается, вычислим апостериорную вероятность темы по формуле  
$P(z \mid d,w) = \dfrac{P(w \mid z) P(z \mid d)}{P(w \mid z=1) P(z=1 \mid d) + P(w \mid z=2) P(z=2 \mid d)}$.

Проведём вычисления для всех 12 пар.

**Для $d_1, w=\text{кошка}$:**  
$P(z=1) = \frac{0.25 \cdot 0.6}{0.25 \cdot 0.6 + 0.03 \cdot 0.4} = \frac{0.15}{0.162} \approx 0.9259$,  
$P(z=2) \approx 0.0741$.

**Для $d_1, w=\text{сидит}$:**  
$P(z=1) = \frac{0.20 \cdot 0.6}{0.20 \cdot 0.6 + 0.05 \cdot 0.4} = \frac{0.12}{0.14} \approx 0.8571$,  
$P(z=2) \approx 0.1429$.

**Для $d_1, w=\text{на}$:**  
$P(z=1) = \frac{0.08 \cdot 0.6}{0.08 \cdot 0.6 + 0.30 \cdot 0.4} = \frac{0.048}{0.168} \approx 0.2857$,  
$P(z=2) \approx 0.7143$.

**Для $d_1, w=\text{окне}$:**  
$P(z=1) = \frac{0.03 \cdot 0.6}{0.03 \cdot 0.6 + 0.25 \cdot 0.4} = \frac{0.018}{0.118} \approx 0.1525$,  
$P(z=2) \approx 0.8475$.

**Для $d_2, w=\text{собака}$:**  
$P(z=1) = \frac{0.25 \cdot 0.3}{0.25 \cdot 0.3 + 0.01 \cdot 0.7} = \frac{0.075}{0.082} \approx 0.9146$,  
$P(z=2) \approx 0.0854$.

**Для $d_2, w=\text{сидит}$:**  
$P(z=1) = \frac{0.20 \cdot 0.3}{0.20 \cdot 0.3 + 0.05 \cdot 0.7} = \frac{0.06}{0.095} \approx 0.6316$,  
$P(z=2) \approx 0.3684$.

**Для $d_2, w=\text{на}$:**  
$P(z=1) = \frac{0.08 \cdot 0.3}{0.08 \cdot 0.3 + 0.30 \cdot 0.7} = \frac{0.024}{0.234} \approx 0.1026$,  
$P(z=2) \approx 0.8974$.

**Для $d_2, w=\text{крыльце}$:**  
$P(z=1) = \frac{0.02 \cdot 0.3}{0.02 \cdot 0.3 + 0.20 \cdot 0.7} = \frac{0.006}{0.146} \approx 0.0411$,  
$P(z=2) \approx 0.9589$.

**Для $d_3, w=\text{кошка}$:**  
$P(z=1) = \frac{0.25 \cdot 0.55}{0.25 \cdot 0.55 + 0.03 \cdot 0.45} = \frac{0.1375}{0.151} \approx 0.9106$,  
$P(z=2) \approx 0.0894$.

**Для $d_3, w=\text{спит}$:**  
$P(z=1) = \frac{0.15 \cdot 0.55}{0.15 \cdot 0.55 + 0.01 \cdot 0.45} = \frac{0.0825}{0.087} \approx 0.9483$,  
$P(z=2) \approx 0.0517$.

**Для $d_3, w=\text{на}$:**  
$P(z=1) = \frac{0.08 \cdot 0.55}{0.08 \cdot 0.55 + 0.30 \cdot 0.45} = \frac{0.044}{0.179} \approx 0.2458$,  
$P(z=2) \approx 0.7542$.

**Для $d_3, w=\text{диване}$:**  
$P(z=1) = \frac{0.02 \cdot 0.55}{0.02 \cdot 0.55 + 0.15 \cdot 0.45} = \frac{0.011}{0.0785} \approx 0.1401$,  
$P(z=2) \approx 0.8599$.

Сведём результаты в таблицу (округление до 4 знаков).

**Таблица E-шага после инициализации**

| Документ | Слово | $P(z=1 \mid d,w)$ | $P(z=2 \mid d,w)$ |
|----------|-------|-------------------|-------------------|
| $d_1$ | кошка | 0.9259 | 0.0741 |
| $d_1$ | сидит | 0.8571 | 0.1429 |
| $d_1$ | на    | 0.2857 | 0.7143 |
| $d_1$ | окне  | 0.1525 | 0.8475 |
| $d_2$ | собака | 0.9146 | 0.0854 |
| $d_2$ | сидит | 0.6316 | 0.3684 |
| $d_2$ | на    | 0.1026 | 0.8974 |
| $d_2$ | крыльце | 0.0411 | 0.9589 |
| $d_3$ | кошка | 0.9106 | 0.0894 |
| $d_3$ | спит  | 0.9483 | 0.0517 |
| $d_3$ | на    | 0.2458 | 0.7542 |
| $d_3$ | диване | 0.1401 | 0.8599 |

#### Итерация 1: M-шаг

Обновим параметры $P(w \mid z)$ и $P(z \mid d)$.

**Обновление $P(w \mid z)$**

Для каждого слова $w$ и темы $z$ числитель равен сумме апостериорных вероятностей $P(z \mid d,w)$ по всем документам, где слово встречается. Знаменатель — сумма всех таких вероятностей для данной темы (по всем 12 парам).

Вычислим числители.

| Слово | $\sum P(z{=}1 \mid d,w)$ | $\sum P(z{=}2 \mid d,w)$ |
|-------|--------------------------|--------------------------|
| кошка | 0.9259 + 0.9106 = 1.8365 | 0.0741 + 0.0894 = 0.1635 |
| сидит | 0.8571 + 0.6316 = 1.4887 | 0.1429 + 0.3684 = 0.5113 |
| на    | 0.2857 + 0.1026 + 0.2458 = 0.6341 | 0.7143 + 0.8974 + 0.7542 = 2.3659 |
| окне  | 0.1525 | 0.8475 |
| собака | 0.9146 | 0.0854 |
| крыльце | 0.0411 | 0.9589 |
| спит  | 0.9483 | 0.0517 |
| диване | 0.1401 | 0.8599 |

Суммы по темам:  
Для $z=1$: $1.8365 + 1.4887 + 0.6341 + 0.1525 + 0.9146 + 0.0411 + 0.9483 + 0.1401 = 6.1559$.  
Для $z=2$: $0.1635 + 0.5113 + 2.3659 + 0.8475 + 0.0854 + 0.9589 + 0.0517 + 0.8599 = 5.8441$.  
(Сумма обеих равна 12, что соответствует числу словоупотреблений.)

Новые вероятности:

**$P(w \mid z=1)$:**

| слово | вероятность |
|-------|-------------|
| кошка | 1.8365 / 6.1559 ≈ 0.2983 |
| сидит | 1.4887 / 6.1559 ≈ 0.2418 |
| на    | 0.6341 / 6.1559 ≈ 0.1030 |
| окне  | 0.1525 / 6.1559 ≈ 0.0248 |
| собака | 0.9146 / 6.1559 ≈ 0.1486 |
| крыльце | 0.0411 / 6.1559 ≈ 0.0067 |
| спит  | 0.9483 / 6.1559 ≈ 0.1541 |
| диване | 0.1401 / 6.1559 ≈ 0.0228 |

**$P(w \mid z=2)$:**

| слово | вероятность |
|-------|-------------|
| кошка | 0.1635 / 5.8441 ≈ 0.0280 |
| сидит | 0.5113 / 5.8441 ≈ 0.0875 |
| на    | 2.3659 / 5.8441 ≈ 0.4048 |
| окне  | 0.8475 / 5.8441 ≈ 0.1450 |
| собака | 0.0854 / 5.8441 ≈ 0.0146 |
| крыльце | 0.9589 / 5.8441 ≈ 0.1641 |
| спит  | 0.0517 / 5.8441 ≈ 0.0088 |
| диване | 0.8599 / 5.8441 ≈ 0.1471 |

**Обновление $P(z \mid d)$**

Для каждого документа сумма апостериорных вероятностей по всем его словам делится на длину документа (4).

Для $d_1$:  
$z=1$: $0.9259 + 0.8571 + 0.2857 + 0.1525 = 2.2212$, $P(z=1 \mid d_1) = 2.2212/4 = 0.5553$;  
$z=2$: $0.0741 + 0.1429 + 0.7143 + 0.8475 = 1.7788$, $P(z=2 \mid d_1) = 1.7788/4 = 0.4447$.

Для $d_2$:  
$z=1$: $0.9146 + 0.6316 + 0.1026 + 0.0411 = 1.6899$, $P(z=1 \mid d_2) = 1.6899/4 = 0.4225$;  
$z=2$: $0.0854 + 0.3684 + 0.8974 + 0.9589 = 2.3101$, $P(z=2 \mid d_2) = 2.3101/4 = 0.5775$.

Для $d_3$:  
$z=1$: $0.9106 + 0.9483 + 0.2458 + 0.1401 = 2.2448$, $P(z=1 \mid d_3) = 2.2448/4 = 0.5612$;  
$z=2$: $0.0894 + 0.0517 + 0.7542 + 0.8599 = 1.7552$, $P(z=2 \mid d_3) = 1.7552/4 = 0.4388$.

#### Вычисление логарифма правдоподобия

**Начальный логарифм правдоподобия** (до EM) с исходными параметрами:

Для каждой пары вычислим $P(w,d) = P(z{=}1|d)P(w|z{=}1) + P(z{=}2|d)P(w|z{=}2)$ и логарифм.

Для $d_1$:  
- кошка: $0.6 \cdot 0.25 + 0.4 \cdot 0.03 = 0.162$, $\ln = -1.820$;  
- сидит: $0.6 \cdot 0.20 + 0.4 \cdot 0.05 = 0.14$, $\ln = -1.966$;  
- на: $0.6 \cdot 0.08 + 0.4 \cdot 0.30 = 0.168$, $\ln = -1.784$;  
- окне: $0.6 \cdot 0.03 + 0.4 \cdot 0.25 = 0.118$, $\ln = -2.136$.  
Сумма $d_1 = -7.706$.

Для $d_2$:  
- собака: $0.3 \cdot 0.25 + 0.7 \cdot 0.01 = 0.082$, $\ln = -2.501$;  
- сидит: $0.3 \cdot 0.20 + 0.7 \cdot 0.05 = 0.095$, $\ln = -2.354$;  
- на: $0.3 \cdot 0.08 + 0.7 \cdot 0.30 = 0.234$, $\ln = -1.453$;  
- крыльце: $0.3 \cdot 0.02 + 0.7 \cdot 0.20 = 0.146$, $\ln = -1.924$.  
Сумма $d_2 = -8.232$.

Для $d_3$:  
- кошка: $0.55 \cdot 0.25 + 0.45 \cdot 0.03 = 0.151$, $\ln = -1.890$;  
- спит: $0.55 \cdot 0.15 + 0.45 \cdot 0.01 = 0.087$, $\ln = -2.442$;  
- на: $0.55 \cdot 0.08 + 0.45 \cdot 0.30 = 0.179$, $\ln = -1.720$;  
- диване: $0.55 \cdot 0.02 + 0.45 \cdot 0.15 = 0.0785$, $\ln = -2.544$.  
Сумма $d_3 = -8.596$.

Общий начальный логарифм правдоподобия:  
$\ell_0 = -7.706 -8.232 -8.596 = -24.534$.

**Логарифм правдоподобия после первой итерации** (с обновлёнными параметрами):

Используем новые $P(w|z)$ и $P(z|d)$, округлённые до 4 знаков.

Для $d_1$:  
- кошка: $0.5553 \cdot 0.2983 + 0.4447 \cdot 0.0280 = 0.1781$, $\ln \approx -1.724$;  
- сидит: $0.5553 \cdot 0.2418 + 0.4447 \cdot 0.0875 = 0.1732$, $\ln \approx -1.753$;  
- на: $0.5553 \cdot 0.1030 + 0.4447 \cdot 0.4048 = 0.2372$, $\ln \approx -1.439$;  
- окне: $0.5553 \cdot 0.0248 + 0.4447 \cdot 0.1450 = 0.0783$, $\ln \approx -2.547$.  
Сумма $d_1 \approx -7.463$.

Для $d_2$:  
- собака: $0.4225 \cdot 0.1486 + 0.5775 \cdot 0.0146 = 0.0712$, $\ln \approx -2.642$;  
- сидит: $0.4225 \cdot 0.2418 + 0.5775 \cdot 0.0875 = 0.1527$, $\ln \approx -1.879$;  
- на: $0.4225 \cdot 0.1030 + 0.5775 \cdot 0.4048 = 0.2773$, $\ln \approx -1.283$;  
- крыльце: $0.4225 \cdot 0.0067 + 0.5775 \cdot 0.1641 = 0.0976$, $\ln \approx -2.327$.  
Сумма $d_2 \approx -8.131$.

Для $d_3$:  
- кошка: $0.5612 \cdot 0.2983 + 0.4388 \cdot 0.0280 = 0.1797$, $\ln \approx -1.716$;  
- спит: $0.5612 \cdot 0.1541 + 0.4388 \cdot 0.0088 = 0.0904$, $\ln \approx -2.404$;  
- на: $0.5612 \cdot 0.1030 + 0.4388 \cdot 0.4048 = 0.2355$, $\ln \approx -1.446$;  
- диване: $0.5612 \cdot 0.0228 + 0.4388 \cdot 0.1471 = 0.0774$, $\ln \approx -2.559$.  
Сумма $d_3 \approx -8.125$.

Общий логарифм правдоподобия после первой итерации:  
$\ell_1 = -7.463 -8.131 -8.125 = -23.719$.

Видно, что $\ell_1 > \ell_0$ ($-23.719 > -24.534$), т.е. правдоподобие возросло.

#### Итерация 2 и последующие

На второй итерации повторяются E-шаг с обновлёнными параметрами и M-шаг. Процесс сходится за несколько итераций. Приведём финальные параметры после 10 итераций (округлённо до 3 знаков; получены численным моделированием).

**Итоговое $P(w|z=1)$:**

| кошка | сидит | на | окне | собака | крыльце | спит | диване |
|-------|-------|----|------|--------|---------|------|--------|
| 0.258 | 0.207 | 0.088 | 0.021 | 0.240 | 0.007 | 0.156 | 0.023 |

**Итоговое $P(w|z=2)$:**

| кошка | сидит | на | окне | собака | крыльце | спит | диване |
|-------|-------|----|------|--------|---------|------|--------|
| 0.021 | 0.043 | 0.421 | 0.156 | 0.002 | 0.177 | 0.009 | 0.171 |

**Итоговое $P(z|d)$:**

| Документ | $P(z=1|d)$ | $P(z=2|d)$ |
|----------|-------------|-------------|
| $d_1$ | 0.613 | 0.387 |
| $d_2$ | 0.405 | 0.595 |
| $d_3$ | 0.582 | 0.418 |

Логарифм правдоподобия после 10 итераций: $\ell \approx -22.9$, что показывает дальнейший рост.


## 4.1.6 Интерпретация результатов

Полученные распределения слов по темам имеют ясную интерпретацию. Тема 1 ($z=1$) выделяет слова, связанные с животными и действиями: наибольшие вероятности у слов «кошка» (0.258), «собака» (0.240), «сидит» (0.207), «спит» (0.156). Тема 2 ($z=2$) акцентирует места и предлоги: «на» (0.421), «крыльце» (0.177), «диване» (0.171), «окне» (0.156). Таким образом, pLSA автоматически разделила лексику на две смысловые группы, хотя ей не было дано никаких явных указаний.

Распределения тем по документам показывают, что документ 1 и документ 3 (про кошку) имеют более высокую вероятность темы 1 (0.613 и 0.582 соответственно), а документ 2 (про собаку) — темы 2 (0.595). Это согласуется с интуицией: документы 1 и 3 содержат слова «кошка», «сидит», «спит» и место (окно, диван), но тема 1 доминирует, а документ 2 про собаку и крыльцо больше связан с темой 2 (места), хотя в нём также есть слово «сидит».

## 4.1.7 Ограничения pLSA

Несмотря на интерпретируемость и статистическую строгость, pLSA имеет два серьёзных недостатка. Во-первых, количество параметров $P(z \mid d)$ растёт линейно с числом документов: для каждого документа обучается свой вектор $\theta_d$. Это приводит к переобучению, особенно на небольших коллекциях, и требует большого объёма данных. Во-вторых, модель не определяет вероятностного распределения для новых документов: если появляется документ, не входивший в обучающую выборку, для него нет параметра $P(z \mid d_{\text{new}})$, и его нужно оценивать заново, удерживая темы фиксированными. Это делает pLSA неудобной для практических приложений, где постоянно поступают новые тексты.

Эти ограничения преодолеваются в модели **латентного размещения Дирихле** (LDA), которая добавляет байесовские априорные распределения на параметры и тем самым решает проблему переобучения и обеспечивает естественный способ обработки новых документов. Переход к LDA будет рассмотрен в следующей части.

# Часть 4.2. Латентное размещение Дирихле (LDA)

## 4.2.1 Байесовское расширение pLSA

Вероятностный латентный семантический анализ, рассмотренный в предыдущей части, позволил моделировать документы как смеси скрытых тем, но обладал двумя принципиальными ограничениями. Во-первых, количество параметров $P(z \mid d)$ растёт линейно с числом документов, что ведёт к переобучению. Во-вторых, модель не задаёт естественного вероятностного механизма для новых документов: для текста, не входившего в обучающую коллекцию, параметры $\theta_d$ не определены, и их приходится оценивать отдельно, удерживая темы фиксированными.

Эти проблемы решаются в модели **латентного размещения Дирихле** (Latent Dirichlet Allocation, LDA), предложенной Блеем, Нг и Джорданом в 2003 году. LDA превращает pLSA в полностью байесовскую модель, добавляя априорные распределения на параметры $\theta_d$ (распределение тем документа) и $\phi_z$ (распределение слов темы). Вместо того чтобы считать эти величины детерминированными параметрами, LDA рассматривает их как случайные векторы, порождённые из распределений Дирихле. Это обеспечивает сглаживание оценок, снижает риск переобучения и позволяет корректно обрабатывать новые документы через апостериорный вывод.

## 4.2.2 Распределение Дирихле

Распределение Дирихле — это многомерное обобщение бета-распределения, определённое на симплексе вероятностных векторов. Вектор $\theta = (\theta_1, \theta_2, \ldots, \theta_K)$ лежит в $(K-1)$-мерном симплексе, то есть $\theta_i \ge 0$ и $\sum_{i=1}^K \theta_i = 1$. Плотность распределения Дирихле с параметром $\alpha = (\alpha_1, \ldots, \alpha_K)$, где $\alpha_i > 0$, задаётся формулой

$$
p(\theta \mid \alpha) = \frac{\Gamma\left(\sum_{i=1}^K \alpha_i\right)}{\prod_{i=1}^K \Gamma(\alpha_i)} \prod_{i=1}^K \theta_i^{\alpha_i - 1},
$$

где $\Gamma$ — гамма-функция. Параметр $\alpha$ управляет формой распределения: при $\alpha_i < 1$ распределение концентрируется на разреженных векторах (большинство компонент близки к нулю, одна доминирует), при $\alpha_i > 1$ — на более равномерных. В LDA обычно используют симметричные скалярные параметры $\alpha$ и $\beta$, одинаковые для всех тем и слов.

Аналогично распределение слов в теме $\phi_z$ имеет априорное распределение Дирихле с параметром $\beta$ (также скалярным, применяемым ко всем словам словаря).

## 4.2.3 Порождающий процесс LDA

Формально порождающий процесс для коллекции из $M$ документов выглядит следующим образом.

1. Для каждой темы $z = 1, \ldots, K$ выбрать распределение слов $\phi_z \sim \text{Dirichlet}(\beta)$.
2. Для каждого документа $d = 1, \ldots, M$:
   - выбрать распределение тем $\theta_d \sim \text{Dirichlet}(\alpha)$;
   - для каждой позиции слова $n = 1, \ldots, N_d$:
     * выбрать тему $z_{d,n} \sim \text{Multinomial}(\theta_d)$;
     * выбрать слово $w_{d,n} \sim \text{Multinomial}(\phi_{z_{d,n}})$.

Здесь $N_d$ — длина документа $d$. В отличие от pLSA, где $\theta_d$ и $\phi_z$ были фиксированными параметрами, в LDA они сами являются случайными величинами. Априорные параметры $\alpha$ и $\beta$ задаются пользователем и обычно малы (например, $\alpha = 0.1$, $\beta = 0.01$), что способствует разреженности.

## 4.2.4 Совместное распределение и задача вывода

Совместное распределение всех наблюдаемых слов $W = \{w_{d,n}\}$, скрытых тем $Z = \{z_{d,n}\}$, параметров $\Theta = \{\theta_d\}$ и $\Phi = \{\phi_z\}$ имеет вид

$$
P(W, Z, \Theta, \Phi \mid \alpha, \beta) = \prod_{z=1}^K P(\phi_z \mid \beta) \prod_{d=1}^M P(\theta_d \mid \alpha) \prod_{n=1}^{N_d} P(z_{d,n} \mid \theta_d) P(w_{d,n} \mid \phi_{z_{d,n}}).
$$

Цель вывода — вычислить апостериорное распределение скрытых переменных $P(Z, \Theta, \Phi \mid W, \alpha, \beta)$. Из-за связи между $\theta$ и $\phi$ через темы это распределение не имеет аналитической формы. На практике применяют два основных подхода: **сэмплирование Гиббса** и **вариационный вывод**. Мы сосредоточимся на сэмплировании Гиббса, которое интуитивно понятно и широко используется.

## 4.2.5 Сэмплирование Гиббса для LDA

Сэмплирование Гиббса — это метод Монте-Карло с марковскими цепями, который итеративно сэмплирует каждую скрытую переменную из её условного распределения при фиксированных остальных переменных. В LDA мы не храним $\theta$ и $\phi$ явно, а интегрируем их, работая только с назначениями тем $z_{d,n}$ для каждого слова. Благодаря сопряжённости Дирихле и мультиномиального распределения, можно вывести простое условное распределение для темы одного слова.

Обозначим через $n_{d,k}^{-(d,n)}$ количество слов в документе $d$, отнесённых к теме $k$, не считая текущее слово $(d,n)$. Аналогично, $n_{k,w}^{-(d,n)}$ — количество раз, когда слово $w$ было связано с темой $k$ во всей коллекции, не считая текущее вхождение. Тогда условное распределение темы $z_{d,n}$ при условии всех остальных тем и наблюдаемых слов имеет вид

$$
P(z_{d,n} = k \mid Z^{-(d,n)}, W, \alpha, \beta) \propto \frac{n_{d,k}^{-(d,n)} + \alpha_k}{\sum_{k'} \left( n_{d,k'}^{-(d,n)} + \alpha_{k'} \right)} \cdot \frac{n_{k,w}^{-(d,n)} + \beta_w}{\sum_{w'} \left( n_{k,w'}^{-(d,n)} + \beta_{w'} \right)}.
$$

При использовании симметричных скалярных $\alpha$ и $\beta$ формула упрощается:

$$
P(z_{d,n} = k \mid \ldots) \propto \left( n_{d,k}^{-(d,n)} + \alpha \right) \cdot \frac{n_{k,w}^{-(d,n)} + \beta}{\sum_{w'} n_{k,w'}^{-(d,n)} + N \beta},
$$

где $N$ — размер словаря. Знаменатель во втором сомножителе не зависит от $k$ при сравнении тем для данного слова, поэтому его можно опустить. Тогда

$$
P(z_{d,n} = k \mid \ldots) \propto \left( n_{d,k}^{-(d,n)} + \alpha \right) \cdot \left( n_{k,w}^{-(d,n)} + \beta \right).
$$

Именно эту пропорциональность используют на практике: для каждого возможного $k$ вычисляют произведение двух счётчиков с добавками $\alpha$ и $\beta$, затем нормализуют, чтобы получить вероятности, и сэмплируют тему.

После достаточного числа итераций (burn-in) сэмплы тем стабилизируются, и можно оценить параметры $\theta$ и $\phi$ по формулам:

$$
\hat{\theta}_{d,k} = \frac{n_{d,k} + \alpha}{\sum_{k'} (n_{d,k'} + \alpha)},
$$

$$
\hat{\phi}_{k,w} = \frac{n_{k,w} + \beta}{\sum_{w'} (n_{k,w'} + \beta)}.
$$










## 4.2.6 Численный пример LDA на учебном корпусе

Вернёмся к нашему корпусу из трёх документов:

- $d_1$: «кошка сидит на окне»
- $d_2$: «собака сидит на крыльце»
- $d_3$: «кошка спит на диване»

Словарь $V$ содержит 8 слов: $N=8$. Выберем число тем $K=2$. Установим гиперпараметры $\alpha = 0.1$, $\beta = 0.01$ (типичные значения). Для простоты будем считать, что каждое слово в каждом документе встречается ровно один раз, поэтому всего 12 слов в коллекции.

### Инициализация тем

Перед началом сэмплирования необходимо присвоить каждому слову случайную начальную тему. Для воспроизводимости зададим конкретное начальное назначение:

- $d_1$: (кошка→1, сидит→1, на→2, окне→2)
- $d_2$: (собака→1, сидит→1, на→2, крыльце→2)
- $d_3$: (кошка→1, спит→1, на→2, диване→2)

Это детерминированное начальное состояние. Теперь подсчитаем счётчики $n_{d,k}$ и $n_{k,w}$.

**Счётчики $n_{d,k}$ (число слов в документе с темой k):**

| Документ | Тема 1 | Тема 2 |
|----------|--------|--------|
| $d_1$ | 2 | 2 |
| $d_2$ | 2 | 2 |
| $d_3$ | 2 | 2 |

**Счётчики $n_{k,w}$ (число вхождений слова w с темой k):**

Для темы 1:
- кошка: $d_1$ (1) + $d_3$ (1) = 2
- сидит: $d_1$ (1) + $d_2$ (1) = 2
- собака: $d_2$ (1) = 1
- спит: $d_3$ (1) = 1
- остальные (на, окне, крыльце, диване) = 0.

Для темы 2:
- на: $d_1$ (1) + $d_2$ (1) + $d_3$ (1) = 3
- окне: $d_1$ (1) = 1
- крыльце: $d_2$ (1) = 1
- диване: $d_3$ (1) = 1
- остальные (кошка, сидит, собака, спит) = 0.

### Первая итерация сэмплирования

Пройдём по каждому слову в порядке документов и позиций, обновляя тему. При обновлении исключаем текущее слово из счётчиков, вычисляем вероятности для каждой темы.

*Важно: в реальном сэмплировании Гиббса тема выбирается **случайно** пропорционально вычисленным вероятностям. В этом учебном примере для простоты и наглядности мы будем выбирать тему с максимальной вероятностью, то есть детерминированно. Это допустимо для иллюстрации, но в практических реализациях всегда используется стохастический выбор.*

#### Слово 1: $d_1$, «кошка», текущая тема 1

Исключаем это слово. Тогда $n_{d_1,1}^{-(1,1)} = 2-1=1$, $n_{1,\text{кошка}}^{-(1,1)} = 2-1=1$.

Для темы $k=1$: $(n_{d_1,1}^{-} + \alpha) \cdot (n_{1,\text{кошка}}^{-} + \beta) = (1+0.1) \cdot (1+0.01) = 1.1 \times 1.01 = 1.111$.

Для темы $k=2$: $(n_{d_1,2}^{-} + \alpha) \cdot (n_{2,\text{кошка}}^{-} + \beta) = (2+0.1) \cdot (0+0.01) = 2.1 \times 0.01 = 0.021$.

Вероятность темы 1: $1.111 / (1.111+0.021) \approx 0.981$, темы 2: $0.019$. Выбираем тему 1, так как её вероятность выше. Счётчики не меняются.

#### Слово 2: $d_1$, «сидит», текущая тема 1

Исключаем. $n_{d_1,1}^{-} = 1$, $n_{1,\text{сидит}}^{-}=2-1=1$.

Для $k=1$: $(1+0.1)\cdot(1+0.01)=1.111$.

Для $k=2$: $n_{d_1,2}^{-}=2$ (на и окне темы 2), $n_{2,\text{сидит}}^{-}=0$, произведение $(2+0.1)\cdot(0+0.01)=0.021$.

Выбираем тему 1. Счётчики не меняются.

#### Слово 3: $d_1$, «на», текущая тема 2

Исключаем. $n_{d_1,2}^{-}=2-1=1$, $n_{2,\text{на}}^{-}=3-1=2$.

Для $k=1$: $n_{d_1,1}^{-}=2$ (кошка и сидит темы 1), $n_{1,\text{на}}^{-}=0$ (слово «на» с темой 1 не встречалось). Произведение $(2+0.1)\cdot(0+0.01)=0.021$.

Для $k=2$: $n_{d_1,2}^{-}=1$, $n_{2,\text{на}}^{-}=2$. Произведение $(1+0.1)\cdot(2+0.01)=1.1\times2.01=2.211$.

Вероятность темы 2: $2.211/(2.211+0.021)\approx0.991$. Оставляем тему 2.

#### Слово 4: $d_1$, «окне», текущая тема 2

Исключаем. $n_{d_1,2}^{-}=1$, $n_{2,\text{окне}}^{-}=1-1=0$.

Для $k=1$: $n_{d_1,1}^{-}=2$, $n_{1,\text{окне}}^{-}=0$ → $(2+0.1)\cdot(0+0.01)=0.021$.

Для $k=2$: $n_{d_1,2}^{-}=1$, $n_{2,\text{окне}}^{-}=0$ → $(1+0.1)\cdot(0+0.01)=0.011$.

Сравним: $0.021 > 0.011$, поэтому выбираем тему 1. Слово «окне» переходит в тему 1. Обновим счётчики: $n_{d_1,1}$ станет 3, $n_{d_1,2}$ станет 1; $n_{1,\text{окне}}$ станет 1, $n_{2,\text{окне}}$ станет 0.

Зафиксируем это изменение. Теперь состояние после обработки $d_1$:
- $d_1$: кошка(1), сидит(1), на(2), окне(1). Итого $n_{d_1,1}=3$, $n_{d_1,2}=1$.
- Счётчики $n_{k,w}$: тема 1: кошка 2, сидит 2, окне 1, собака 1, спит 1; тема 2: на 3, крыльце 1, диване 1 (окне 0). Проверим суммы: тема 1 всего $2+2+1+1+1=7$, тема 2 $3+1+1=5$, всего 12.

#### Слово 5: $d_2$, «собака», текущая тема 1

Исключаем. $n_{d_2,1}^{-}=1$ (сидит), $n_{1,\text{собака}}^{-}=1-1=0$.

Для $k=1$: $(1+0.1)\cdot(0+0.01)=0.011$.
Для $k=2$: $n_{d_2,2}^{-}=2$ (на, крыльце), $n_{2,\text{собака}}^{-}=0$ → $(2+0.1)\cdot(0+0.01)=0.021$.

Тема 2 имеет большую вероятность. Переводим «собака» в тему 2. Обновим: $n_{d_2,1}=1$, $n_{d_2,2}=3$; $n_{1,\text{собака}}=0$, $n_{2,\text{собака}}=1$.

#### Слово 6: $d_2$, «сидит», текущая тема 1

Исключаем. $n_{d_2,1}^{-}=0$, $n_{1,\text{сидит}}^{-}=2-1=1$.

Для $k=1$: $(0+0.1)\cdot(1+0.01)=0.101$.
Для $k=2$: $n_{d_2,2}^{-}=3$ (собака, на, крыльце), $n_{2,\text{сидит}}^{-}=0$ → $(3+0.1)\cdot(0+0.01)=0.031$.

Тема 1 имеет большую вероятность. Оставляем тему 1.

#### Слово 7: $d_2$, «на», текущая тема 2

Исключаем. $n_{d_2,2}^{-}=3-1=2$ (собака, крыльце), $n_{2,\text{на}}^{-}=3-1=2$.

Для $k=1$: $n_{d_2,1}^{-}=1$ (сидит), $n_{1,\text{на}}^{-}=0$ → $(1+0.1)\cdot(0+0.01)=0.011$.
Для $k=2$: $n_{d_2,2}^{-}=2$, $n_{2,\text{на}}^{-}=2$ → $(2+0.1)\cdot(2+0.01)=4.221$.

Остаётся тема 2.

#### Слово 8: $d_2$, «крыльце», текущая тема 2

Исключаем. $n_{d_2,2}^{-}=2$ (собака, на), $n_{2,\text{крыльце}}^{-}=1-1=0$.

Для $k=1$: $n_{d_2,1}^{-}=1$ (сидит), $n_{1,\text{крыльце}}^{-}=0$ → $(1+0.1)\cdot(0+0.01)=0.011$.
Для $k=2$: $n_{d_2,2}^{-}=2$, $n_{2,\text{крыльце}}^{-}=0$ → $(2+0.1)\cdot(0+0.01)=0.021$.

Тема 2 имеет большую вероятность (0.021 > 0.011). Оставляем тему 2.

После $d_2$ состояние:
- $d_2$: собака(2), сидит(1), на(2), крыльце(2). Итого $n_{d_2,1}=1$, $n_{d_2,2}=3$.
- Счётчики $n_{k,w}$: тема 1: кошка 2, сидит 2, окне 1, спит 1 (всего 6). Тема 2: на 3, крыльце 1, диване 1, собака 1 (всего 6). Сумма 12.

#### Слово 9: $d_3$, «кошка», текущая тема 1

Исключаем. $n_{d_3,1}^{-}=1$ (спит), $n_{1,\text{кошка}}^{-}=2-1=1$.

Для $k=1$: $(1+0.1)\cdot(1+0.01)=1.111$.
Для $k=2$: $n_{d_3,2}^{-}=2$ (на, диване), $n_{2,\text{кошка}}^{-}=0$ → $(2+0.1)\cdot(0+0.01)=0.021$.

Оставляем тему 1.

#### Слово 10: $d_3$, «спит», текущая тема 1

Исключаем. $n_{d_3,1}^{-}=1$ (кошка), $n_{1,\text{спит}}^{-}=1-1=0$.

Для $k=1$: $(1+0.1)\cdot(0+0.01)=0.011$.
Для $k=2$: $n_{d_3,2}^{-}=2$, $n_{2,\text{спит}}^{-}=0$ → $(2+0.1)\cdot(0+0.01)=0.021$.

Тема 2 имеет большую вероятность. Переводим «спит» в тему 2. Обновим: $n_{d_3,1}=1$, $n_{d_3,2}=3$; $n_{1,\text{спит}}=0$, $n_{2,\text{спит}}=1$.

#### Слово 11: $d_3$, «на», текущая тема 2

Исключаем. $n_{d_3,2}^{-}=3-1=2$ (диване, спит), $n_{2,\text{на}}^{-}=3-1=2$.

Для $k=1$: $n_{d_3,1}^{-}=1$ (кошка), $n_{1,\text{на}}^{-}=0$ → $(1+0.1)\cdot(0+0.01)=0.011$.
Для $k=2$: $n_{d_3,2}^{-}=2$, $n_{2,\text{на}}^{-}=2$ → $(2+0.1)\cdot(2+0.01)=4.221$.

Остаётся тема 2.

#### Слово 12: $d_3$, «диване», текущая тема 2

Исключаем. $n_{d_3,2}^{-}=2$ (спит, на), $n_{2,\text{диване}}^{-}=1-1=0$.

Для $k=1$: $n_{d_3,1}^{-}=1$ (кошка), $n_{1,\text{диване}}^{-}=0$ → $(1+0.1)\cdot(0+0.01)=0.011$.
Для $k=2$: $n_{d_3,2}^{-}=2$, $n_{2,\text{диване}}^{-}=0$ → $(2+0.1)\cdot(0+0.01)=0.021$.

Тема 2 более вероятна, оставляем.

После первой полной итерации состояние:

**Назначения тем:**
- $d_1$: кошка(1), сидит(1), на(2), окне(1) → $n_{d_1,1}=3$, $n_{d_1,2}=1$.
- $d_2$: собака(2), сидит(1), на(2), крыльце(2) → $n_{d_2,1}=1$, $n_{d_2,2}=3$.
- $d_3$: кошка(1), спит(2), на(2), диване(2) → $n_{d_3,1}=1$, $n_{d_3,2}=3$.

**Счётчики $n_{k,w}$:**
- Тема 1: кошка 2, сидит 2, окне 1 (всего 5).
- Тема 2: на 3, крыльце 1, диване 1, собака 1, спит 1 (всего 7).

Проверка: всего 12 слов, 5+7=12.

### Вторая итерация (кратко)

На второй итерации повторяем проход по всем словам с использованием новых счётчиков. Покажем несколько примеров, чтобы проиллюстрировать сходимость.

#### Слово 1: $d_1$, «кошка», тема 1

Исключаем. $n_{d_1,1}^{-}=3-1=2$, $n_{1,\text{кошка}}^{-}=2-1=1$.

Для $k=1$: $(2+0.1)\cdot(1+0.01)=2.1\times1.01=2.121$.
Для $k=2$: $n_{d_1,2}^{-}=1$ (на), $n_{2,\text{кошка}}^{-}=0$ → $(1+0.1)\cdot(0+0.01)=0.011$.

Вероятность темы 1 почти 1. Оставляем.

#### Слово 4: $d_1$, «окне», тема 1

Исключаем. $n_{d_1,1}^{-}=2$ (кошка, сидит), $n_{1,\text{окне}}^{-}=1-1=0$.

Для $k=1$: $(2+0.1)\cdot(0+0.01)=0.021$.
Для $k=2$: $n_{d_1,2}^{-}=1$ (на), $n_{2,\text{окне}}^{-}=0$ → $(1+0.1)\cdot(0+0.01)=0.011$.

Тема 1 чуть более вероятна (0.021 vs 0.011), но это уже почти случайность. Для простоты предположим, что остаётся тема 1.

Остальные слова, вероятно, сохранят свои темы, так как структура уже стабилизировалась. *В нашем учебном примере из-за крошечного размера корпуса назначения тем действительно быстро перестают меняться. В реальных задачах для стабилизации цепи обычно требуется значительно больше итераций (например, сотни или тысячи), и процесс сходимости отслеживают по стабилизации логарифма правдоподобия или другим диагностикам.*

После нескольких итераций состояние сходится к следующему (округлённо):

**Итоговое назначение тем:**
- $d_1$: кошка(1), сидит(1), на(2), окне(1) (тема 1: 3, тема 2: 1)
- $d_2$: собака(2), сидит(1), на(2), крыльце(2) (тема 1: 1, тема 2: 3)
- $d_3$: кошка(1), спит(2), на(2), диване(2) (тема 1: 1, тема 2: 3)

**Итоговые счётчики $n_{k,w}$:**
- Тема 1: кошка 2, сидит 2, окне 1 (всего 5).
- Тема 2: на 3, крыльце 1, диване 1, собака 1, спит 1 (всего 7).

(Это состояние совпадает с концом первой итерации, что говорит о быстрой сходимости для такого маленького корпуса.)

## 4.2.7 Оценка параметров $\theta$ и $\phi$

Используем формулы с учётом счётчиков и гиперпараметров.

**Оценка $\theta_{d,k} = \frac{n_{d,k} + \alpha}{\sum_{k'} (n_{d,k'} + \alpha)}$.**

Для $d_1$: $n_{d_1,1}=3$, $n_{d_1,2}=1$, $\alpha=0.1$.

$$
\theta_{d_1,1} = \frac{3 + 0.1}{4 + 2 \times 0.1} = \frac{3.1}{4.2} \approx 0.738,
$$
$$
\theta_{d_1,2} = \frac{1 + 0.1}{4.2} \approx 0.262.
$$

Для $d_2$: $n_{d_2,1}=1$, $n_{d_2,2}=3$.

$$
\theta_{d_2,1} = \frac{1.1}{4.2} \approx 0.262,
$$
$$
\theta_{d_2,2} = \frac{3.1}{4.2} \approx 0.738.
$$

Для $d_3$: аналогично $d_2$, так как счётчики такие же.

$$
\theta_{d_3,1} \approx 0.262, \quad \theta_{d_3,2} \approx 0.738.
$$

**Оценка $\phi_{k,w} = \frac{n_{k,w} + \beta}{\sum_{w'} (n_{k,w'} + \beta)}$, где $\beta=0.01$.**

Для темы 1: сумма по всем словам $n_{1,w'} = 5$, плюс $8 \times 0.01 = 0.08$. Знаменатель $5 + 8\cdot 0.01 = 5.08$.

- кошка: $n_{1,\text{кошка}} = 2$, $\hat{\phi} = (2+0.01)/5.08 = 2.01/5.08 \approx 0.396$.
- сидит: $2.01/5.08 \approx 0.396$.
- окне: $1.01/5.08 \approx 0.199$.
- остальные (собака, на, крыльце, спит, диване): $0.01/5.08 \approx 0.002$.

Для темы 2: сумма $n_{2,w'} = 7$, знаменатель $7 + 0.08 = 7.08$.

- на: $3.01/7.08 \approx 0.425$.
- крыльце: $1.01/7.08 \approx 0.143$.
- диване: $1.01/7.08 \approx 0.143$.
- собака: $1.01/7.08 \approx 0.143$.
- спит: $1.01/7.08 \approx 0.143$.
- остальные (кошка, сидит, окне): $0.01/7.08 \approx 0.001$.

## 4.2.8 Интерпретация и сравнение с pLSA

Полученные темы имеют ясную интерпретацию. Тема 1 сконцентрирована на словах «кошка» (0.396), «сидит» (0.396), «окне» (0.199), что соответствует контексту «кошка сидит на окне». Тема 2 выделяет «на» (0.425), а также равномерно «крыльце», «диване», «собака», «спит» (по 0.143). Это несколько иная группировка, чем в pLSA, где тема 1 объединяла животных и действия, а тема 2 — места и предлоги. В LDA из-за малого размера корпуса и конкретной инициализации темы получились более смешанными: тема 1 сфокусировалась на связке «кошка сидит на окне» (документ 1), а тема 2 — на остальных словах, включая «собака» и «спит», которые в pLSA относились к теме 1. Это объясняется тем, что LDA — вероятностная модель с априорными ограничениями, и на крошечных данных результат может зависеть от начального состояния и случайности сэмплирования.

Однако ключевое преимущество LDA перед pLSA видно в оценках $\theta_d$: благодаря добавлению $\alpha$ и $\beta$ параметры сглажены, и ни одно значение не обращается в нуль, даже для слов, не встретившихся в теме. Например, слово «собака» в теме 1 имеет ненулевую вероятность (0.002), хотя в обучающих данных не встречалось с темой 1. Это обеспечивает устойчивость к переобучению и позволяет модели обобщать на новые документы. В pLSA без сглаживания вероятности для не встреченных комбинаций могли бы быть нулевыми.

Кроме того, LDA способна обрабатывать новые документы: зафиксировав обученные $\phi_z$, можно вывести распределение тем $\theta_{\text{new}}$ для нового текста с помощью нескольких дополнительных итераций сэмплирования или вариационного вывода. В pLSA для этого требовалось бы переобучение параметров документа с нуля, но глобальные темы были бы фиксированы, что делало процесс не вполне байесовским.


## 4.2.9 Заключение по LDA

Латентное размещение Дирихле стало стандартом тематического моделирования благодаря своей статистической строгости, способности к обобщению и естественной интерпретируемости. Она устраняет недостатки pLSA, вводя байесовские априорные распределения, которые сглаживают оценки и позволяют корректно работать с новыми текстами. В практических приложениях LDA применяется для анализа больших коллекций документов, выявления скрытых тем, кластеризации и снижения размерности.

Мы рассмотрели вывод с помощью сэмплирования Гиббса, который интуитивно понятен и широко используется в реализации. Существуют и другие методы, такие как вариационный вывод и онлайн-варианты, обеспечивающие масштабируемость на огромные корпуса. Понимание математических основ LDA необходимо для осознанного применения модели и её расширений, таких как динамические тематические модели, коррелированные тематические модели и нейротематические подходы.

На этом завершается обзор классических методов векторизации текста — от one-hot encoding до LDA. Каждый из этих подходов вносил вклад в решение проблемы числового представления естественного языка, постепенно приближая нас к современным нейросетевым эмбеддингам, которые будут рассмотрены в следующих лекциях.